# Interative Pauli functions

In [8]:
# %pip install ipycytoscape ipywidgets  # run once in Jupyter if needed

import ipycytoscape as cy
import ipywidgets as W
from ipywidgets import Layout
import numpy as np
from typing import Any, List, Tuple, Dict, Optional, Callable, Union, Iterable, Sequence
from ipyforcegraph.graphs import ForceGraph3D
from collections import defaultdict
import cmath
import matplotlib.pyplot as plt

def pauli_list_to_dict(pauli_list):
    pauli_dict = defaultdict(float)
    for coeff, pauli in pauli_list:
        pauli_dict[pauli] += coeff
    return pauli_dict

def pauli_dict_to_list(pauli_dict):
    pauli_list = []
    for pauli, coeff in pauli_dict.items():
        pauli_list.append((coeff, pauli))
    return pauli_list

def clean_pauli_list(pauli_list):
    return pauli_dict_to_list(pauli_list_to_dict(pauli_list))

# ---------- small utils ----------
def _popcount(x: int) -> int:
    c = 0
    while x:
        x &= x - 1
        c += 1
    return c

def _bitstring(u: int, n: int, indexing="lsb") -> str:
    # MSB → LSB to match left-to-right Pauli order
    if indexing == "lsb":
        return format(u, f"0{n}b")
    elif indexing == "msb":
        return format(u, f"0{n}b")[::-1]
    else:
        return ValueError("indexing must be lsb or msb")

def hamming_weight_equals(k: int) -> Callable[[int], bool]:
    return lambda u: _popcount(u) == k

# Layered (Hamming-weight) initial positions: concentric circles
def _layered_positions(n: int, r_step: float = 120.0) -> Dict[int, tuple]:
    """
    Place nodes with Hamming weight k on a circle of radius k * r_step.
    Angle order within a layer is by node index.
    """
    pos = {}
    for k in range(n+1):
        layer = [u for u in range(1 << n) if _popcount(u) == k]
        m = max(1, len(layer))
        for j, u in enumerate(sorted(layer)):
            ang = 2.0 * np.pi * j / m
            r = r_step * k
            pos[u] = (float(r * np.cos(ang)), float(r * np.sin(ang)))
    return pos

# ---------- Pauli action (LEFT char ↔ MSB) ----------
def _flip_parity_from_pstring(pstr: str):
    """
    LEFTMOST Pauli char acts on MSB (bit index n-1).
    P|u> = i^{nY} * (-1)^{popcount(u & parity)} |u XOR flip|.
    """
    n = len(pstr)
    flip = 0; nY = 0; parity = 0
    for j, P in enumerate(pstr):
        bit = n - 1 - j  # MSB-aligned
        if P == 'X':
            flip |= (1 << bit)
        elif P == 'Y':
            flip |= (1 << bit); nY += 1; parity |= (1 << bit)
        elif P == 'Z':
            parity |= (1 << bit)
        elif P != 'I':
            raise ValueError("Pauli string must contain only I,X,Y,Z")
    return flip, nY, parity

'''
# Build undirected pairs with per-label complex amplitudes; drop canceled pairs (FIXED)
def build_pairs_by_label(pauli_terms: List[Tuple[complex, str]],
                         cancel_tol: float = 1e-12):
    if not pauli_terms:
        raise ValueError("Empty pauli_terms")
    n = len(pauli_terms[0][1])
    if any(len(p) != n for _, p in pauli_terms):
        raise ValueError("All Pauli strings must have equal length")
    N = 1 << n

    pair_net: Dict[tuple, complex] = {}
    pair_term: Dict[tuple, Dict[str, complex]] = {}

    for c, pstr in pauli_terms:
        flip, nY, parity = _flip_parity_from_pstring(pstr)
        if flip == 0:
            continue  # diagonal-only; no off-diagonal edge
        phaseY = (1j) ** nY
        for u in range(N):
            v = u ^ flip
            if u >= v:   # undirected (use each pair once)
                continue
            phase_par = -1 if (_popcount(u & parity) & 1) else 1
            amp = c * phaseY * phase_par
            pair_net[(u, v)] = pair_net.get((u, v), 0.0) + amp
            d = pair_term.setdefault((u, v), {})   # <-- create/get dict first
            d[pstr] = d.get(pstr, 0.0) + amp       # <-- then update it

    # Keep only pairs whose net doesn't cancel
    survivors = {pair: tm for pair, tm in pair_term.items()
                 if abs(pair_net.get(pair, 0.0)) > cancel_tol}
    return survivors, n
'''

def build_pairs_by_label(pauli_terms, cancel_tol: float = 1e-12,
                         keep_zero_pairs: bool = False,    # NEW
                         per_label_tol: float = 1e-12):    # NEW
    if not pauli_terms:
        raise ValueError("Empty pauli_terms")
    n = len(pauli_terms[0][1])
    if any([len(p) != n for _, p in pauli_terms]):
        raise ValueError("All Pauli strings must have equal length")
    N = 1 << n

    pair_net: Dict[tuple, complex] = {}
    pair_term: Dict[tuple, Dict[str, complex]] = {}

    for c, pstr in pauli_terms:
        flip, nY, parity = _flip_parity_from_pstring(pstr)
        if flip == 0:
            continue
        phaseY = (1j) ** nY
        for u in range(N):
            v = u ^ flip
            if u >= v:
                continue
            phase_par = -1 if (_popcount(u & parity) & 1) else 1
            amp = c * phaseY * phase_par
            pair_net[(u, v)] = pair_net.get((u, v), 0.0) + amp
            d = pair_term.setdefault((u, v), {})
            d[pstr] = d.get(pstr, 0.0) + amp

    # in build_pairs_by_label(...)
    survivors: Dict[tuple, Dict[str, complex]] = {}
    for pair, tm in pair_term.items():
        # prune tiny per-label contributions, keep labels that matter
        tm_pruned = {lab: a for lab, a in tm.items() if abs(a) > per_label_tol}
        if not tm_pruned:
            continue
        if abs(pair_net.get(pair, 0.0)) > cancel_tol:   # keep only if net ≠ 0
            survivors[pair] = tm_pruned

    return survivors, n

# Color map in FIRST-occurrence order of input pauli_terms
def palette_from_input_order(pauli_terms: List[Tuple[complex, str]]):
    labels_in_input = [p for _, p in pauli_terms]
    label_to_index = {}
    label_order = []
    for lab in labels_in_input:
        if lab not in label_to_index:
            label_to_index[lab] = len(label_order)
            label_order.append(lab)
    # 20 qualitative colors (tab20)
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap('tab20')
    def color_for(lab: str) -> str:
        idx = label_to_index[lab]
        r,g,b,a = cmap((idx % 20)/19.0)
        return f"#{int(255*r):02x}{int(255*g):02x}{int(255*b):02x}"
    return color_for, label_order, label_to_index

# ---------- partitioned Hamming-weight helpers ----------
def _segment_masks(n: int, m: int) -> List[int]:
    """Return m masks for m equal MSB→LSB partitions (requires n % m == 0)."""
    if n % m != 0:
        raise ValueError(f"n={n} must be divisible by number of partitions m={m}")
    seg = n // m
    masks = []
    for s in range(m):
        low  = n - (s+1)*seg    # inclusive bit index
        high = n - s*seg - 1    # inclusive bit index
        width = high - low + 1
        mask = ((1 << width) - 1) << low
        masks.append(mask)
    return masks

def partition_weight_filter(n: int, weights: Sequence[int]) -> Callable[[int], bool]:
    """
    Keep nodes whose bitstring has specified Hamming weight per equal partition.
    weights: (k1, k2, ..., km) from LEFT (MSB) to RIGHT (LSB).
    Requires n % m == 0 with m=len(weights).
    """
    m = len(weights)
    seg = n // m
    for idx, k in enumerate(weights):
        if not (0 <= k <= seg):
            raise ValueError(f"Partition {idx}: 0 <= k <= {seg} required, got {k}")
    masks = _segment_masks(n, m)
    def include(u: int) -> bool:
        for k, mask in zip(weights, masks):
            if _popcount(u & mask) != k:
                return False
        return True
    return include

# ---------- Main 2D widget (draggable) ----------
from collections import defaultdict
from typing import Dict, Iterable, List, Tuple

def pauli_terms_to_basis_elements(
    n: int,
    pauli_terms: Iterable[Tuple[complex, str]],
    *,
    indexing: str = "lsb",
    drop_diagonal: bool = True,
    tol: float = 1e-12,
) -> Dict[Tuple[int, int], complex]:
    """
    Expand a sum of Pauli strings into individual computational-basis matrix elements.

    Returns a dict: (u, v) -> coeff  where u, v in [0, 2^n).
    If `drop_diagonal=True`, only off-diagonal elements are returned.

    Conventions (computational basis |0>,|1>):
      I: <b|I|b> = 1
      Z: <b|Z|b> = +1 if b=0 else -1 (diagonal only)
      X: <b^1|X|b> = 1  (flip)
      Y: <b^1|Y|b> = i  if b=0,  -i if b=1  (flip with phase i*(+1 for 0, -1 for 1))

    `indexing` affects only how you *display* bitstrings elsewhere; the math here is bitwise.
    """
    # Precompute per-qubit operator masks and types for each Pauli string
    def _bit(b: int, k: int) -> int:
        return (b >> k) & 1

    def _flip_mask(pstr: str) -> int:
        # Bits where we flip (X or Y)
        m = 0
        for k, ch in enumerate(pstr):
            if ch in ("X", "Y"):
                m |= (1 << k)
        return m

    # Accumulate (u, v) contributions
    acc = defaultdict(complex)
    N = 1 << n

    for coeff, pstr in pauli_terms:
        pstr = pstr.strip()
        if len(pstr) != n:
            raise ValueError(f"Pauli string '{pstr}' has length {len(pstr)} but n={n}.")

        flip_mask = _flip_mask(pstr)

        # If there are only I/Z, we contribute (possibly) diagonal elements only.
        # If any X/Y present, we contribute off-diagonals only (no diagonal support).
        if flip_mask == 0:
            if drop_diagonal:
                # Still need to handle diagonal if requested off, just skip.
                continue
            # Diagonal: <u|P|u> = coeff * Π_k factor_k(u_k), with
            #   I->1, Z->(+1 if 0 else -1)
            for u in range(N):
                phase = 1.0 + 0.0j
                # multiply Z signs
                ok = True
                for k, ch in enumerate(pstr):
                    if ch == "Z":
                        phase *= (1.0 if _bit(u, k) == 0 else -1.0)
                    elif ch == "X" or ch == "Y":
                        # Shouldn't happen because flip_mask==0
                        ok = False
                        break
                if not ok:
                    continue
                val = coeff * phase
                if abs(val) > tol:
                    acc[(u, u)] += val
            continue

        # Off-diagonal support: for each basis state u, v = u ^ flip_mask
        # Phase comes from Z on non-flipped bits and Y on flipped bits.
        for u in range(N):
            v = u ^ flip_mask

            # If we only want off-diagonals, ensure u != v (it always is here since flip_mask!=0)
            # but keep the check (robustness if someone passes weird inputs).
            if drop_diagonal and u == v:
                continue

            phase = 1.0 + 0.0j
            ok = True
            for k, ch in enumerate(pstr):
                b = _bit(u, k)
                if ch == "I":
                    pass
                elif ch == "Z":
                    # Requires no flip on this qubit, which is satisfied (flip only on X/Y positions)
                    phase *= (1.0 if b == 0 else -1.0)
                elif ch == "X":
                    # flip handled by v = u ^ flip_mask; matrix element magnitude 1
                    pass
                elif ch == "Y":
                    # flip with phase: i if b=0 else -i
                    phase *= (1j if b == 0 else -1j)
                else:
                    ok = False
                    break
            if not ok:
                continue

            val = coeff * phase
            if abs(val) > tol:
                acc[(u, v)] += val

    # Drop tiny
    return {uv: z for uv, z in acc.items() if abs(z) > tol}


def format_edge_label(u: int, v: int, n: int, *, indexing: str = "lsb") -> str:
    """
    Render an edge label like 'b... -> b'...' suitable for display.
    Honors 'lsb' (qubit 0 is rightmost) or 'msb' (qubit 0 is leftmost) conventions.
    """
    def _bitstring(x: int) -> str:
        s = f"{x:0{n}b}"
        return s if indexing == "msb" else s[::-1]
    return f"{_bitstring(u)} → {_bitstring(v)}"


from collections import defaultdict
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Union, Any
import cmath, numpy as np
import matplotlib.pyplot as plt
import ipycytoscape as cy
import ipywidgets as W
from ipywidgets import Layout

# assumes you already have:
# _bitstring, _popcount, _layered_positions, _char_pos,
# build_pairs_by_label, palette_from_input_order, partition_weight_filter

from typing import Iterable, Tuple, Union, Optional, Dict, Any, List
from collections import defaultdict
import cmath, numpy as np
import matplotlib.pyplot as plt

_LOCAL_ELL_TO_PAIR = {
    0: (0,1),  # 00 <-> 01
    1: (2,3),  # 10 <-> 11
    2: (0,2),  # 00 <-> 10
    3: (1,3),  # 01 <-> 11
    4: (0,3),  # 00 <-> 11
    5: (1,2),  # 01 <-> 10
}

def _pair_str_to_ints(s: str) -> Tuple[int,int]:
    s = s.strip().replace('"','').replace("'",'')
    parts = s.split()
    if len(parts) != 2 or any(p not in ("00","01","10","11") for p in parts):
        raise ValueError(f'Bad local pair string: {s!r}')
    m = {"00":0,"01":1,"10":2,"11":3}
    return m[parts[0]], m[parts[1]]

def _ints_to_pair_str(a: int, b: int) -> str:
    m = {0:"00",1:"01",2:"10",3:"11"}
    # keep the exact order (don’t sort) to preserve caller’s intent
    return f"{m[a]} {m[b]}"

def _normalize_local_triples(
    triples: Iterable[Union[Tuple[int,int,int], Tuple[int,int,int,complex], Tuple[int,int,str], Tuple[int,int,str,complex]]],
    *,
    n_override: Optional[int] = None,
) -> Tuple[List[Tuple[int,int,int,int,complex,str]], int]:
    """
    Returns: list of (i, j, aa, bb, amp, label), and n (#qubits).
    If n_override is None, infer n = 1 + max(i,j) over triples.
    """
    out: List[Tuple[int,int,int,int,complex,str]] = []
    maxq = -1

    for t in triples:
        if len(t) == 3:
            i, j, ell = t
            amp = 1.0
        elif len(t) == 4:
            i, j, ell, amp = t
        else:
            raise ValueError(f"Each triple must be (i,j,ell) or (i,j,ell,amp); got {t}")

        maxq = max(maxq, i, j)
        # ell may be int 0..5 or a string like "00 11"
        if isinstance(ell, str):
            aa, bb = _pair_str_to_ints(ell)
            label = f"({i},{j}) {_ints_to_pair_str(aa, bb)}"
        else:
            if ell not in _LOCAL_ELL_TO_PAIR:
                raise ValueError(f"ell must be 0..5 or 'aa bb', got {ell}")
            aa, bb = _LOCAL_ELL_TO_PAIR[ell]
            label = f"({i},{j}) {_ints_to_pair_str(aa, bb)}"

        out.append((int(i), int(j), int(aa), int(bb), complex(amp), label))

    n = n_override if n_override is not None else (maxq + 1 if maxq >= 0 else 0)
    if n <= 0:
        raise ValueError("Cannot infer n; provide at least one triple or set n_override.")
    return out, n

def _pair_bits(u: int, i: int, j: int) -> int:
    return ((_bit(u, i) << 1) | _bit(u, j))  # 0..3

def _swap_pair(u: int, i: int, j: int, new_pair: int) -> int:
    # write bits back to positions i (MSB of pair), j (LSB of pair)
    ui = (new_pair >> 1) & 1
    uj = new_pair & 1
    # clear those bits then set
    u2 = u & ~(1 << i) & ~(1 << j)
    if ui: u2 |= (1 << i)
    if uj: u2 |= (1 << j)
    return u2

def _logical_to_phys(i: int, n: int, indexing: str) -> int:
    # logical i -> physical bit position in integer u
    if indexing == "lsb":
        return i
    elif indexing == "msb":
        return (n - 1 - i)
    else:
        raise ValueError(f"indexing must be 'lsb' or 'msb', got {indexing!r}")

def _bit_at(u: int, phys: int) -> int:
    return (u >> phys) & 1

def _pair_bits_indexed(u: int, i: int, j: int, n: int, indexing: str) -> int:
    pi = _logical_to_phys(i, n, indexing)
    pj = _logical_to_phys(j, n, indexing)
    # keep the previous convention: pair = (bit at i as MSB, bit at j as LSB)
    return ((_bit_at(u, pi) << 1) | _bit_at(u, pj))

def _swap_pair_indexed(u: int, i: int, j: int, n: int, new_pair: int, indexing: str) -> int:
    pi = _logical_to_phys(i, n, indexing)
    pj = _logical_to_phys(j, n, indexing)
    bi = (new_pair >> 1) & 1  # MSB goes to i
    bj = (new_pair     ) & 1  # LSB goes to j
    # clear both bits
    u2 = u & ~(1 << pi) & ~(1 << pj)
    # set as needed
    if bi: u2 |= (1 << pi)
    if bj: u2 |= (1 << pj)
    return u2

def build_local_survivors_from_triples(
    n: int,
    triples_norm: List[Tuple[int,int,int,int,complex,str]],
    *,
    indexing: str = "lsb",
    tol: float = 1e-12,
):
    """
    survivors_local[(u,v)][label] = sum of complex amps from the provided local triples,
    where (i,j) refers to logical qubits and is mapped to bit positions using `indexing`.
    """
    N = 1 << n
    survivors_local = defaultdict(lambda: defaultdict(complex))
    label_order: List[str] = []
    seen = set()

    for (i, j, aa, bb, amp, label) in triples_norm:
        if label not in seen:
            seen.add(label); label_order.append(label)
        for u in range(N):
            if _pair_bits_indexed(u, i, j, n, indexing) != aa:
                continue
            v = _swap_pair_indexed(u, i, j, n, bb, indexing)
            if u == v:
                continue
            survivors_local[(u, v)][label] += complex(amp)

    # prune tiny
    for key in list(survivors_local.keys()):
        d = survivors_local[key]
        for lbl in list(d.keys()):
            if abs(d[lbl]) <= tol:
                del d[lbl]
        if not d:
            del survivors_local[key]

    return survivors_local, label_order


def pauli_cyto_widget(
    pauli_terms: List[Tuple[complex, str]],
    cancel_tol: float = 1e-12,
    hamming_k: Optional[int] = None,
    keep_zero_pairs: bool = False,
    per_label_tol: float = 1e-12,
    width_by_ampl: bool = True,
    base_width: float = 2.0,
    always_curve: bool = False,
    curve_step: float = 10.0,
    node_size: float = 22.0,
    node_color_mode: Optional[str] = "particle num",
    label_font: int = 12,
    r_step: float = 120.0,
    partition_weights: Optional[Sequence[int]] = None,
    layout_name: str = "preset",
    layout_kwargs: Optional[Dict[str, Any]] = None,
    opt_passes: int = 0,
    height: str = "500px",

    # --- Weights ---
    width_from_weight: bool = False,
    width_ref: float = 1.0,
    show_weights: bool = True,
    show_labels: bool = True,
    weight_mode: str = "cartesian",
    weight_label_mode: str = "both",   # used in 'pauli' mode
    weight_prec: int = 3,
    weight_zero_tol: float = 1e-12,
    
    # --- Line graph ---
    enforce_line: bool = False,
    line_gap: float = 72,
    
    # --- Bipartite ---
    enforce_bipartite: bool = False,
    bipartite_gap: float = 240.0,
    bipartite_vspace: float = 72.0,
    bipartite_group_by_partile_num = True,
    bipartite_even_rev = False,
    bipartite_odd_rev = True,

    # --- NEW: input mode ---
    input_mode: str = "pauli",     # 'pauli' (existing) or 'local' (NEW)
    local_triples: Optional[Iterable[Union[Tuple[int,int,int], Tuple[int,int,int,complex], Tuple[int,int,str], Tuple[int,int,str,complex]]]] = None,
    n_override: Optional[int] = None,
    indexing: str = "lsb",  # kept for your existing element formatting helper if you need it elsewhere
    
    # --- Pair-wise penalties ---
    penalty_pairs: List[Tuple[int, int, str]] = None,
):
    # ---------------- decide how to build survivors & n ----------------
    if input_mode == "local":
        if not local_triples:
            raise ValueError("input_mode='local' requires local_triples.")
        triples_norm, n = _normalize_local_triples(local_triples, n_override=n_override)
        survivors_local, label_order_local = build_local_survivors_from_triples(
            n, triples_norm, indexing=indexing, tol=per_label_tol   # <— pass indexing here
        )
        survivors = None  # unused in this mode
    else:
        # your existing path
        survivors, n = build_pairs_by_label(
            pauli_terms,
            cancel_tol=cancel_tol,
            keep_zero_pairs=keep_zero_pairs,
            per_label_tol=per_label_tol
        )

    N = 1 << n

    # -------- visibility/filtering (unchanged) --------
    visible = set(range(N)) if hamming_k is None else {u for u in range(N) if _popcount(u) == hamming_k}
    if partition_weights is not None:
        include = partition_weight_filter(n, partition_weights)
        visible &= {u for u in range(N) if include(u)}
        
    if penalty_pairs is not None:
        for i, j, pstr in penalty_pairs: # remove penalty pairs
            # Parse forbidden patterns, e.g. "00 01" -> {"00", "01"}
            forbidden_patterns = set(pstr.split())

            # Collect indices to remove for this (i, j) pair
            to_remove = set()

            for idx in visible:
                b_i = (idx >> (n-i-1)) & 1
                b_j = (idx >> (n-j-1)) & 1
                pattern = f"{b_i}{b_j}"

                if pattern in forbidden_patterns:
                    to_remove.add(idx)

            # Remove all forbidden indices at once
            visible -= to_remove

    # -------- positions (unchanged, but we print visible in your code) --------
    if enforce_line and (not enforce_bipartite):
        vis = sorted(visible)
        pos = {u: (i*line_gap, 0) for (i, u) in enumerate(vis)}
    elif enforce_bipartite and (not enforce_line):
        vis = sorted(visible)
        even = [u for u in vis if (_popcount(u) % 2) == 0]
        odd  = [u for u in vis if (_popcount(u) % 2) == 1]
        if bipartite_group_by_partile_num:
            even = sorted(even, key=_popcount, reverse=bipartite_even_rev)
            odd  = sorted(odd,  key=_popcount, reverse=bipartite_odd_rev)
        xL, xR = -bipartite_gap/2.0, bipartite_gap/2.0
        def _stack_y(lst):
            m = len(lst)
            return {u: (idx - (m - 1)/2.0) * bipartite_vspace for idx, u in enumerate(lst)}
        y_even = _stack_y(even); y_odd = _stack_y(odd)
        pos = {u: (xL, y_even[u]) for u in even}
        pos.update({u: (xR, y_odd[u]) for u in odd})
    elif enforce_bipartite and enforce_line:
        raise ValueError("Cannot be both line and bipartite graph")
    else:
        pos = _layered_positions(n, r_step=r_step)

    # -------- widget + style (kept) --------
    G = cy.CytoscapeWidget()
    G.layout = Layout(height=height, width='100%')
    G.set_style([
        {'selector': 'node',
         'style': {
            'label': 'data(label)',
            'width': node_size, 'height': node_size,
            'background-color': '#111',
            'color': 'data(color)',
            'font-size': f'{label_font}px',
            'text-valign': 'top',
            'text-halign': 'center',
            'text-margin-y': '-12px',
         }},
        {'selector': 'edge',
         'style': {
            'curve-style': 'unbundled-bezier',
            'edge-distances': 'node-position',
            'control-point-distances': 'data(cp_dists)',
            'control-point-weights':   'data(cp_wgts)',
            'line-color': 'data(color)',
            'width': 'data(width)',
            'opacity': 0.95,
            'line-cap': 'round',
            'label': 'data(label_disp)',
            'text-rotation': 'autorotate',
            'font-size': 10,
            'text-background-opacity': 1,
            'text-background-color': '#ffffff',
            'text-background-shape': 'roundrectangle',
            'text-margin-y': -2,
         }},
    ])

    # node colors by Hamming weight (kept)
    cmap = plt.get_cmap('tab20')
    def idx_color_for(idx: int) -> str:
        if node_color_mode == "pnum":
            r,g,b,a = cmap(idx*2)
            return f"#{int(255*r):02x}{int(255*g):02x}{int(255*b):02x}"
        elif node_color_mode == None:
            return None
    nodes = []
    for u in range(N):
        if u not in visible: 
            continue
        x, y = pos[u]
        nodes.append({
            'data': {'id': str(u), 
                    'label': _bitstring(u, n),
                    'color': idx_color_for(_popcount(u))},
            'position': {'x': x, 'y': y}
        })

    def _fmt_complex(z: complex) -> str:
        eps = weight_zero_tol
        re0, im0 = abs(z.real) <= eps, abs(z.imag) <= eps
        if weight_mode == "polar":
            r, th = abs(z), cmath.phase(z)
            return f"{r:.{weight_prec}g}" if r <= eps else f"{r:.{weight_prec}g} ∠ {th:.{weight_prec}g}"
        if re0 and im0: return "0"
        if im0:         return f"{z.real:.{weight_prec}g}"
        if re0:         return f"{z.imag:.{weight_prec}g}i"
        sgn = "+" if z.imag >= 0 else "-"
        return f"{z.real:.{weight_prec}g} {sgn} {abs(z.imag):.{weight_prec}g}i"

    edges = []

    if input_mode == "local":
        # palette for local labels
        def local_color_for(lbl: str) -> str:
            # use label order for color indexing
            idx = label_order_local.index(lbl)
            r,g,b,a = cmap(2*n + 2*(idx % cmap.N))
            return f"#{int(255*r):02x}{int(255*g):02x}{int(255*b):02x}"

        # width reference from local magnitudes
        amps = [abs(z) for tm in survivors_local.values() for z in tm.values()]
        max_amp = max(amps) if amps else 1.0

        # straight edge if the pair has only one edge
        per_pair_counts = defaultdict(int)
        for (u, v), termmap in survivors_local.items():
            if (u not in visible) or (v not in visible):
                continue
            per_pair_counts[(u, v)] += len(termmap)

        for (u, v), termmap in survivors_local.items():
            if (u not in visible) or (v not in visible):
                continue

            items = sorted(termmap.items(), key=lambda kv: label_order_local.index(kv[0]))
            k = len(items)
            offsets = np.arange(k) - (k - 1)/2.0
            only_one = (per_pair_counts[(u, v)] == 1)

            for idx, (lbl, coeff) in enumerate(items):
                c = complex(coeff)
                mag = abs(c)
                width = base_width * (0.5 + 0.5*(mag/max_amp)) if width_by_ampl else base_width
                if width_from_weight:
                    scale = mag / max(1e-12, float(width_ref))
                    width = min(10.0, max(0.5, width * scale))

                if always_curve:
                    cp_dist = float(curve_step * (1.0 + 0.18*abs(offsets[idx])) * (1 if (idx % 2 == 0) else -1))
                else:
                    cp_dist = 0.0 if only_one else float(
                        curve_step * (1.0 + 0.18*abs(offsets[idx])) * (1 if (idx % 2 == 0) else -1)
                    )

                if show_weights:
                    wtxt = _fmt_complex(c)
                    label_disp = f"{lbl} {wtxt}" if show_labels else wtxt
                else:
                    label_disp = lbl if show_labels else ""

                edges.append({'data': {
                    'id': f'{u}-{v}-{lbl}',
                    'source': str(u),
                    'target': str(v),
                    'label': lbl,
                    'label_disp': label_disp,
                    'coeff_eff_re': float(c.real),
                    'coeff_eff_im': float(c.imag),
                    'coeff_abs': float(mag),
                    'coeff_phase': float(cmath.phase(c)),
                    'color': local_color_for(lbl),
                    'width': float(width),
                    'cp_dists': [float(cp_dist)],
                    'cp_wgts':  [0.5],
                }})

        legend_order = label_order_local
        legend_title = "<b>Legend (Local channels):</b><br>"
        legend_color_for = (lambda lab: local_color_for(lab))

    else:
        # -------- your existing PAULI mode (unchanged) --------
        color_for, label_order, label_to_index = palette_from_input_order(pauli_terms)

        amps = [abs(z) for tm in survivors.values() for z in tm.values()]
        max_amp = max(amps) if amps else 1.0

        global_coeff = defaultdict(complex)
        for c,p in pauli_terms:
            global_coeff[p] += complex(c)

        for (u, v), termmap in survivors.items():
            if (u not in visible) or (v not in visible):
                continue

            items = sorted(termmap.items(), key=lambda kv: label_to_index[kv[0]])
            k = len(items)
            offsets = np.arange(k) - (k - 1)/2.0
            only_one = (k == 1)  # per your current rule

            for idx, (lab, coeff_eff) in enumerate(items):
                c_eff = complex(coeff_eff)
                c_alg = global_coeff.get(lab, 0.0+0.0j)
                mag = abs(c_eff)
                width = base_width * (0.5 + 0.5*(mag/max_amp)) if width_by_ampl else base_width
                if width_from_weight:
                    scale = mag / max(1e-12, float(width_ref))
                    width = min(10.0, max(0.5, width * scale))
                cp_dist = 0.0 if only_one else float(
                    curve_step * (1.0 + 0.18*abs(offsets[idx])) * (1 if (idx % 2 == 0) else -1)
                )

                if show_weights:
                    if weight_label_mode == "algebraic":
                        weight_str = _fmt_complex(c_alg)
                        label_disp = f"{lab} {weight_str}"
                    elif weight_label_mode == "effective":
                        weight_str = _fmt_complex(c_eff)
                        label_disp = f"{lab} {weight_str}"
                    else:  # 'both'
                        label_disp = f"{_fmt_complex(c_alg)} {lab} | eff {_fmt_complex(c_eff)}"
                elif show_labels:
                    label_disp = lab
                else:
                    label_disp = ""

                edges.append({'data': {
                    'id': f'{u}-{v}-{lab}',
                    'source': str(u),
                    'target': str(v),
                    'label': lab,
                    'label_disp': label_disp,
                    'coeff_alg_re': float(c_alg.real),
                    'coeff_alg_im': float(c_alg.imag),
                    'coeff_eff_re': float(c_eff.real),
                    'coeff_eff_im': float(c_eff.imag),
                    'coeff_abs': float(mag),
                    'coeff_phase': float(cmath.phase(c_eff)),
                    'color': color_for(lab),
                    'width': float(width),
                    'cp_dists': [float(cp_dist)],
                    'cp_wgts':  [0.5],
                }})

        legend_order = [lab for lab in label_order if any(lab in tm for tm in survivors.values())]
        legend_title = "<b>Legend (Pauli strings):</b><br>"
        legend_color_for = (lambda lab: color_for(lab))

    # ---- add to widget; keep preset and draggable, no auto-relax ----
    G.set_layout(name='preset', randomize=False, animate=False)
    G.graph.add_graph_from_json({'nodes': nodes, 'edges': edges}, multiple_edges=True)
    G.set_layout(name='preset', randomize=False, animate=False)

    # ---- Legend (switch title & palette per mode) ----
    present = legend_order
    legend_html = "<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'>"
    legend_html += legend_title
    for lab in present:
        color = legend_color_for(lab)
        legend_html += f"<span style='display:inline-block;width:14px;height:14px;background:{color};border-radius:2px;margin-right:6px;vertical-align:middle;'></span>{lab}&nbsp;&nbsp;"
    legend_html += "</div>"
    legend = W.HTML(legend_html)

    def get_positions() -> Dict[int, tuple]:
        return {int(n_.data['id']): (float(n_.position['x']), float(n_.position['y'])) for n_ in G.graph.nodes}

    # ---- keep your relax/freeze API (still disabled when bipartite) ----
    def _default_layout_cfg(name: str) -> Dict[str, Any]:
        if name.lower() == 'cose':
            return dict(numIter=8000, coolingFactor=0.99, minTemp=1e-4,
                        randomize=False, animate=False,
                        nodeRepulsion=400000, idealEdgeLength=60, edgeElasticity=100)
        if name.lower() == 'fcose':
            return dict(quality='proof', randomize=False, animate=False,
                        idealEdgeLength=70, nodeRepulsion=4500,
                        gravity=0.25, gravityRange=3.8, gravityCompound=1.0,
                        nodeSeparation=30, packComponents=True)
        if name.lower() == 'cola':
            return dict(maxSimulationTime=60000, infinite=False, randomize=False,
                        animate=False, edgeLength=80, nodeSpacing=20, avoidOverlap=True)
        return {}

    _cfg_base = _default_layout_cfg(layout_name)
    if layout_kwargs:
        _cfg_base.update(layout_kwargs)
    if layout_name.lower() != 'preset':
        _cfg_base.setdefault('randomize', False)
        _cfg_base.setdefault('animate', False)

    def relax_layout(passes: int = 1, **overrides):
        if enforce_bipartite or enforce_line:
            return
        cfg = dict(_cfg_base); cfg.update(overrides)
        cfg['randomize'] = False; cfg['animate'] = False
        for _ in range(max(1, passes)):
            G.set_layout(name=layout_name, **cfg)

    def freeze_layout():
        G.set_layout(name='preset')

    if opt_passes and layout_name.lower() != 'preset' and not enforce_bipartite and not enforce_line:
        relax_layout(passes=opt_passes)

    setattr(G, 'relax_layout', relax_layout)
    setattr(G, 'freeze_layout', freeze_layout)

    return G, legend, get_positions

# ---------- term builders ----------
Number = Union[float, complex]

def _char_pos(q: int, n: int, indexing: str) -> int:
    """Position of qubit q inside the Pauli string of length n."""
    if indexing not in ("msb", "lsb"):
        raise ValueError("indexing must be 'msb' or 'lsb'")
    # 'msb': q==0 is leftmost; 'lsb': q==0 is rightmost
    return q if indexing == "msb" else (n - 1 - q)

def xy_pauli_terms(
    n: int,
    pairs: Iterable[Tuple[int, int]],
    coeff: Union[Number, Dict[Tuple[int,int], Number]] = 1.0,
    indexing: str = "msb",
) -> List[Tuple[complex, str]]:
    """
    Build pauli_terms for an XY Hamiltonian consisting of (XX + YY) on the given pairs.
    """
    out: List[Tuple[complex, str]] = []
    for (i, j) in pairs:
        if not (0 <= i < n and 0 <= j < n):
            raise ValueError(f"pair {(i,j)} out of range for n={n}")
        if i == j:
            continue
        key = (min(i, j), max(i, j))
        J = coeff.get(key, coeff.get((key[1], key[0]), 1.0)) if isinstance(coeff, dict) else coeff
        p = ["I"] * n
        p[_char_pos(i, n, indexing)] = "X"; p[_char_pos(j, n, indexing)] = "X"
        out.append((complex(J), "".join(p)))
        p[_char_pos(i, n, indexing)] = "Y"; p[_char_pos(j, n, indexing)] = "Y"
        out.append((complex(J), "".join(p)))
    return out

def iz_pauli_terms(
    n: int,
    pairs: Iterable[Tuple[int, int]],
    coeff: Union[Number, Dict[Tuple[int,int], Number]] = 1.0,
    indexing: str = "msb",
    include_identity: bool = True,
    aggregate_identity: bool = True,
) -> List[Tuple[complex, str]]:
    """
    Build pauli_terms for (II + ZZ) on the given *pairs* (acting on those two qubits).
    """
    out: List[Tuple[complex, str]] = []
    Istr = "I" * n
    total_I = 0+0j

    for (i, j) in pairs:
        if not (0 <= i < n and 0 <= j < n):
            raise ValueError(f"pair {(i,j)} out of range for n={n}")
        if i == j:
            continue
        key = (min(i, j), max(i, j))
        K = coeff.get(key, coeff.get((key[1], key[0]), 1.0)) if isinstance(coeff, dict) else coeff

        # ZZ on the specified pair
        p = ["I"] * n
        p[_char_pos(i, n, indexing)] = "Z"
        p[_char_pos(j, n, indexing)] = "Z"
        out.append((complex(K), "".join(p)))

        # II (global identity) for that pair
        if include_identity:
            if aggregate_identity:
                total_I += complex(K)
            else:
                out.append((complex(K), Istr))

    if include_identity and aggregate_identity and abs(total_I) != 0:
        out.insert(0, (total_I, Istr))  # put identity first for readability

    return out

# ---------- rotatable 3D Plotly view ----------
def _layered_positions_3d(n: int, r_step: float = 1.0):
    """
    3D analogue of _layered_positions:
    put Hamming-weight k states on a sphere of radius k*r_step using a Fibonacci pattern.
    """
    pos = {}
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))
    for k in range(n + 1):
        layer = [u for u in range(1 << n) if _popcount(u) == k]
        m = max(1, len(layer))
        r = r_step * k
        for j, u in enumerate(sorted(layer)):
            y = 1.0 - 2.0 * ((j + 0.5) / m)
            r_xy = np.sqrt(max(0.0, 1.0 - y * y))
            theta = golden_angle * j
            x = np.cos(theta) * r_xy
            z = np.sin(theta) * r_xy
            pos[u] = (float(r * x), float(r * y), float(r * z))
    return pos

def _bezier3d_samples(p0, p1, bend: float, sign: int = 1, n: int = 40):
    """
    Quadratic Bezier in 3D with control point offset along a normal to the chord (p0->p1).
    """
    p0 = np.asarray(p0, float); p1 = np.asarray(p1, float)
    v = p1 - p0
    L = np.linalg.norm(v)
    if L == 0:
        X = np.full(n, p0[0]); Y = np.full(n, p0[1]); Z = np.full(n, p0[2])
        return X, Y, Z
    d = v / L
    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(d, up)) > 0.9:  # avoid near-parallel
        up = np.array([0.0, 1.0, 0.0])
    nvec = np.cross(d, up)
    nlen = np.linalg.norm(nvec)
    if nlen == 0:
        nvec = np.array([1.0, 0.0, 0.0]); nlen = 1.0
    nvec /= nlen
    m = 0.5 * (p0 + p1)
    c = m + sign * bend * L * nvec
    t = np.linspace(0.0, 1.0, n); u = 1.0 - t
    B = (u*u)[:,None]*p0 + (2*u*t)[:,None]*c + (t*t)[:,None]*p1
    return B[:,0], B[:,1], B[:,2]

def pauli_plotly_3d(
    pauli_terms,
    cancel_tol: float = 1e-12,
    hamming_k: Optional[int] = None,              # optional global Hamming filter
    partition_weights: Optional[Tuple[int, ...]] = None,  # optional equal MSB→LSB partitions
    height: int = 750,
    width: int = 950,
    node_size: int = 4,
    node_color: str = "#111111",
    label_font_size: int = 10,
    curve: float = 0.0,           # 0 → straight edges; >0 → curved edges around nodes
    r_step: float = 1.0,           # radial spacing between Hamming shells
    n_curve_samples: int = 40,
    base_line_width: float = 3.0,  # one width per label trace
    title: Optional[str] = "3D Pauli-colored adjacency",
    show: bool = False,            # return fig by default

    # ---- NEW optimization controls ----
    optimize: bool = True,         # run a quick 3D force optimization first
    iterations: int = 200,         # spring layout iters
    blend: float = 0.6,            # 0→keep shells, 1→use pure spring layout
    k: Optional[float] = None,     # ideal edge length (None lets NX choose)
    seed: Optional[int] = 42,      # layout rng seed
):
    """
    Build a rotatable 3D Plotly figure (orbit with mouse). Colors edges by Pauli string;
    bitstrings labeled at nodes. Now optionally runs a 3D spring-layout optimization
    starting from layered-shell seed and blends back toward shells for readability.
    """
    import plotly.graph_objects as go

    survivors, n = build_pairs_by_label(pauli_terms, cancel_tol)
    N = 1 << n

    color_for, label_order, label_to_index = palette_from_input_order(pauli_terms)

    # visibility (global weight)
    visible = set(range(N)) if hamming_k is None else {u for u in range(N) if _popcount(u) == hamming_k}

    # optional partitioned filter (equal partitions)
    if partition_weights is not None:
        m = len(partition_weights)
        if n % m != 0:
            raise ValueError(f"n={n} must be divisible by number of partitions m={m}")
        seg = n // m
        masks = []
        for s in range(m):
            low  = n - (s+1)*seg
            high = n - s*seg - 1
            width_bits = high - low + 1
            mask = ((1 << width_bits) - 1) << low
            masks.append(mask)
        def include(u: int) -> bool:
            for k_req, mask in zip(partition_weights, masks):
                if _popcount(u & mask) != k_req:
                    return False
            return True
        visible &= {u for u in range(N) if include(u)}

    # positions in 3D (seed = layered shells)
    pos3 = _layered_positions_3d(n, r_step=r_step)

    # ---- NEW: quick 3D optimization (NetworkX spring layout) ----
    if optimize:
        try:
            import networkx as nx
            Gnx = nx.Graph()
            Gnx.add_nodes_from(visible)
            # one edge per (u,v) pair if it survives (ignore per-label multiplicity)
            for (u, v), _termmap in survivors.items():
                if u in visible and v in visible:
                    Gnx.add_edge(u, v)
            # seed only visible nodes
            seed_pos = {u: np.array(pos3[u], dtype=float) for u in visible}
            # run 3D spring layout
            L = nx.spring_layout(Gnx, dim=3, pos=seed_pos, iterations=iterations, seed=seed, k=k)

            # scale layout to roughly match seed magnitude
            r_seed = np.mean([np.linalg.norm(seed_pos[u]) for u in visible]) + 1e-12
            r_layout = np.mean([np.linalg.norm(L[u]) for u in visible]) + 1e-12
            scale = r_seed / r_layout

            # blend: keep some shell structure while improving crossings
            for u in visible:
                p_seed = seed_pos[u]
                p_opt  = np.asarray(L[u]) * scale
                pos3[u] = tuple((1.0 - blend) * p_seed + blend * p_opt)
        except Exception:
            # If networkx missing or fails, silently fall back to seeded shells
            pass

    # nodes (visible only)
    Xn, Yn, Zn, texts = [], [], [], []
    for u in sorted(visible):
        x, y, z = pos3[u]
        Xn.append(x); Yn.append(y); Zn.append(z)
        texts.append(_bitstring(u, n))

    node_trace = go.Scatter3d(
        x=Xn, y=Yn, z=Zn,
        mode="markers+text",
        text=texts,
        textfont=dict(size=label_font_size, color="#222"),
        marker=dict(size=node_size, color=node_color),
        hoverinfo="text",
        name="states",
        showlegend=False,
    )

    # edges: one trace per Pauli label (performance)
    label_traces = {}
    for lab in [l for l in label_order if any(l in tm for tm in survivors.values())]:
        label_traces[lab] = dict(x=[], y=[], z=[])

    for (u, v), termmap in survivors.items():
        if (u not in visible) or (v not in visible):
            continue
        p0 = pos3[u]; p1 = pos3[v]
        items = sorted(termmap.items(), key=lambda kv: label_to_index[kv[0]])
        k_overlap = len(items)
        offsets = np.arange(k_overlap) - (k_overlap - 1)/2.0
        for t_off, (lab, _amp) in zip(offsets, items):
            sign = 1 if t_off >= 0 else -1
            bend_eff = curve * (1.0 + 0.18*abs(t_off))
            X, Y, Z = _bezier3d_samples(p0, p1, bend=bend_eff, sign=sign, n=n_curve_samples)
            lt = label_traces[lab]
            lt["x"].extend(list(X) + [None])
            lt["y"].extend(list(Y) + [None])
            lt["z"].extend(list(Z) + [None])

    edge_traces = []
    for lab, coords in label_traces.items():
        edge_traces.append(
            go.Scatter3d(
                x=coords["x"], y=coords["y"], z=coords["z"],
                mode="lines",
                line=dict(color=color_for(lab), width=base_line_width),
                hoverinfo="skip",
                name=lab,
                showlegend=True
            )
        )

    fig = go.Figure(data=edge_traces + [node_trace])
    fig.update_layout(
        width=width, height=height,
        title=title,
        margin=dict(l=0, r=0, b=0, t=36),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data"
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0)
    )

    if show:
        fig.show()
    return fig

def pauli_forcegraph_3d(
    pauli_terms,
    cancel_tol: float = 1e-12,
    hamming_k: Optional[int] = None,
    partition_weights: Optional[Tuple[int, ...]] = None,
    r_step: float = 1.0,
    height: str = '800px',
    width: str = '100%',
    node_rel_size: float = 10.0,   # bigger, obvious nodes
    link_width: float = 2.0,
    curve: float = 0.0,
    debug: bool = False,
    auto_zoom: bool = True,        # NEW: try to frame the view
):
    from ipyforcegraph.graphs import ForceGraph3D

    survivors, n = build_pairs_by_label(pauli_terms, cancel_tol)
    N = 1 << n

    color_for, label_order, label_to_index = palette_from_input_order(pauli_terms)

    # Visibility filters
    visible = set(range(N)) if hamming_k is None else {u for u in range(N) if _popcount(u) == hamming_k}
    if partition_weights is not None:
        m = len(partition_weights)
        if n % m != 0:
            raise ValueError(f"n={n} must be divisible by number of partitions m={m}")
        seg = n // m
        masks = []
        for s in range(m):
            low  = n - (s+1)*seg
            high = n - s*seg - 1
            width_bits = high - low + 1
            mask = ((1 << width_bits) - 1) << low
            masks.append(mask)
        def include(u: int) -> bool:
            for k_req, mask in zip(partition_weights, masks):
                if _popcount(u & mask) != k_req:
                    return False
            return True
        visible &= {u for u in range(N) if include(u)}

    if debug:
        print(f"[forcegraph] n={n}, visible_nodes={len(visible)}, surviving_pairs={len(survivors)}")

    # Initial positions on Hamming shells (3D)
    pos3 = _layered_positions_3d(n, r_step=r_step)

    # Nodes (add 'val' so spheres render for all builds)
    nodes = []
    for u in sorted(visible):
        x, y, z = pos3[u]
        nodes.append({
            "id": str(u),
            "name": _bitstring(u, n),
            "x": float(x), "y": float(y), "z": float(z),
            "color": "#111111",
            "val": 1.0,                 # <-- important for some versions
            "opacity": 1.0
        })

    # Links
    links = []
    for (u, v), termmap in survivors.items():
        if (u not in visible) or (v not in visible):
            continue
        items = sorted(termmap.items(), key=lambda kv: label_to_index[kv[0]])
        k_overlap = len(items)
        if k_overlap <= 1 or curve == 0.0:
            for lab, _amp in items:
                links.append({
                    "source": str(u), "target": str(v),
                    "color": color_for(lab),
                    "label": lab,
                    "width": float(link_width),
                })
        else:
            offsets = np.arange(k_overlap) - (k_overlap - 1)/2.0
            for idx, (lab, _amp) in enumerate(items):
                links.append({
                    "source": str(u), "target": str(v),
                    "color": color_for(lab),
                    "label": lab,
                    "width": float(link_width),
                    "curvature": float(curve * (1.0 + 0.18*abs(offsets[idx]))),
                    "curveRotation": float(0.5 * np.pi * np.sign(offsets[idx])),
                })

    if debug:
        print(f"[forcegraph] nodes_drawn={len(nodes)}, links_drawn={len(links)}")

    FG = ForceGraph3D()
    FG.layout = Layout(height=height, width=width)
    FG.background_color = "#ffffff"

    # Assign graph FIRST, then set accessors (some front-ends apply better this way)
    FG.graph = {"nodes": nodes, "links": links}

    # Accessors / styling
    FG.node_val = "val"            # <-- ensures spheres get a radius
    FG.node_rel_size = node_rel_size
    FG.node_color = "color"
    FG.link_color = "color"
    FG.link_width = link_width
    FG.node_opacity = 1.0
    FG.enable_node_drag = True
    FG.show_tooltip = True
    # Curved links if available
    try:
        FG.link_curvature = "curvature"
        FG.link_curve_rotation = "curveRotation"
    except Exception:
        pass

    # Try to frame the scene
    if auto_zoom:
        try:
            FG.zoom_to_fit(800, 60)  # duration(ms), padding(px)
        except Exception:
            pass

    # Legend (input order, present labels only)
    present = [lab for lab in label_order if any(lab in tm for tm in survivors.values())]
    legend_html = "<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'>"
    legend_html += "<b>Legend (Pauli strings):</b><br>"
    for lab in present:
        color = color_for(lab)
        legend_html += f"<span style='display:inline-block;width:14px;height:14px;background:{color};border-radius:2px;margin-right:6px;vertical-align:middle;'></span>{lab}&nbsp;&nbsp;"
    legend_html += "</div>"
    legend = W.HTML(legend_html)

    return FG, legend

# --- Save 'parts:v1' with automatic component splitting (compatible with your load_parts) ---

import json, gzip, networkx as nx
from typing import Any, Dict, List, Union, Sequence, Optional
from networkx.readwrite import json_graph

def _best_label_for_node(node, data: Dict[str, Any]) -> str:
    def is_bits(s: str) -> bool:
        return isinstance(s, str) and len(s) > 0 and set(s) <= {"0","1"}
    if "label" in data and is_bits(data["label"]): return str(data["label"])
    if "bit"   in data and is_bits(data["bit"]):   return str(data["bit"])
    if isinstance(node, str) and is_bits(node):    return node
    return str(node)

def _canon_int_graph_and_labels(G: Union[nx.Graph, nx.MultiGraph],
                                given_labels: Optional[Dict[Union[int,str], str]] = None) -> Dict[str, Any]:
    """Relabel nodes → 0..N-1 (sorted order) and return labels dict keyed by the new ints."""
    try:
        nodes_sorted = sorted(G.nodes())
    except TypeError:
        nodes_sorted = sorted(G.nodes(), key=lambda x: str(x))
    mapping = {old: i for i, old in enumerate(nodes_sorted)}
    node_data = {u: dict(G.nodes[u]) for u in G.nodes()}
    G_int = nx.relabel_nodes(G, mapping, copy=True)
    labels: Dict[int, str] = {}
    for old, new in mapping.items():
        if given_labels is not None and old in given_labels:
            labels[new] = str(given_labels[old])
        elif given_labels is not None and isinstance(old, int) and str(old) in given_labels:
            labels[new] = str(given_labels[str(old)])
        else:
            labels[new] = _best_label_for_node(old, node_data.get(old, {}))
    return {"G_int": G_int, "labels": labels}

def networkx_from_cyto_widget(G_widget) -> nx.MultiGraph:
    """Convert ipycytoscape.CytoscapeWidget → MultiGraph with bitstring node labels."""
    Gx = nx.MultiGraph()
    id2label: Dict[int, str] = {}
    for n in G_widget.graph.nodes:
        int_id = int(n.data["id"])
        lab = n.data.get("label", str(int_id))
        id2label[int_id] = lab
        Gx.add_node(lab, int_id=int_id, label=lab)
    for e in G_widget.graph.edges:
        s_id = int(e.data["source"]); t_id = int(e.data["target"])
        u = id2label.get(s_id, str(s_id))
        v = id2label.get(t_id, str(t_id))
        attrs = {
            "id": e.data.get("id"),
            "label": e.data.get("label"),
            "color": e.data.get("color"),
            "width": float(e.data.get("width", 1.0)),
            "cp_dist": float(e.data.get("cp_dist", 0.0)),
        }
        Gx.add_edge(u, v, **attrs)
    return Gx

def _coerce_to_graph_and_labels(obj) -> Dict[str, Any]:
    """
    Accept Graph/MultiGraph, CytoscapeWidget, or dict {'graph': G, 'labels': {...}}.
    Returns {'graph': nx.Graph|nx.MultiGraph, 'labels': Optional[dict]}.
    """
    # ipycytoscape widget?
    try:
        import ipycytoscape as _cy  # noqa
        if isinstance(obj, _cy.CytoscapeWidget):
            return {"graph": networkx_from_cyto_widget(obj), "labels": None}
    except Exception:
        pass

    if isinstance(obj, (nx.Graph, nx.MultiGraph)):
        return {"graph": obj, "labels": None}

    if isinstance(obj, dict) and "graph" in obj:
        G = obj["graph"]
        if isinstance(G, (nx.Graph, nx.MultiGraph)):
            return {"graph": G, "labels": obj.get("labels")}
        # allow a widget inside the dict too
        try:
            import ipycytoscape as _cy2  # noqa
            if isinstance(G, _cy2.CytoscapeWidget):
                return {"graph": networkx_from_cyto_widget(G), "labels": obj.get("labels")}
        except Exception:
            pass
        raise TypeError("parts[i]['graph'] must be a NetworkX graph or CytoscapeWidget.")
    raise TypeError("Each part must be a NetworkX Graph/MultiGraph, CytoscapeWidget, or {'graph':..., 'labels':...} dict.")

def _split_components(G: Union[nx.Graph, nx.MultiGraph]) -> List[nx.Graph]:
    """
    Return a list of subgraphs (copies), one per connected component.
    Works for Graph and MultiGraph. Includes isolated nodes as 1-node components.
    """
    comps = []
    for nodes in nx.connected_components(G):
        comps.append(G.subgraph(nodes).copy())
    return comps

def save_parts(parts: Union[Any, Sequence[Any]],
               path: str,
               *,
               split_components: bool = True) -> None:
    """
    Save in format your load_parts(path) expects:
      {
        "format": "parts:v1",
        "components": [
          {"graph": <node-link JSON>, "labels": {int_id: "bitstring"}}, ...
        ]
      }

    - 'parts' can be a single widget/graph/dict or a list mixing them.
    - If split_components=True, each connected component becomes its own entry.
    """
    if not isinstance(parts, (list, tuple)):
        parts = [parts]

    components_out: List[Dict[str, Any]] = []

    for obj in parts:
        coerced = _coerce_to_graph_and_labels(obj)
        G_in = coerced["graph"]
        given_labels = coerced["labels"]

        graphs_to_save = _split_components(G_in) if split_components else [G_in]

        # To keep a stable order, sort components by (-edge_count, -node_count, min(node-name as str))
        def _comp_key(Gc):
            try:
                mn = min(Gc.nodes())
            except TypeError:
                mn = min(map(str, Gc.nodes()))
            return (-Gc.number_of_edges(), -Gc.number_of_nodes(), str(mn))

        for Gc in sorted(graphs_to_save, key=_comp_key):
            # restrict given_labels to this component's nodes if provided
            labels_sub = None
            if given_labels is not None:
                labels_sub = {k: v for k, v in given_labels.items() if k in Gc.nodes() or str(k) in Gc.nodes()}

            canon = _canon_int_graph_and_labels(Gc, labels_sub)
            G_int, labels = canon["G_int"], canon["labels"]
            g_json = json_graph.node_link_data(G_int)
            components_out.append({
                "graph": g_json,
                "labels": {int(k): str(v) for k, v in labels.items()},
            })

    payload = {"format": "parts:v1", "components": components_out}
    if path.endswith(".gz"):
        with gzip.open(path, "wt", encoding="utf-8") as fh:
            json.dump(payload, fh, separators=(",", ":"))
    else:
        with open(path, "w", encoding="utf-8") as fh:
            json.dump(payload, fh, separators=(",", ":"))

from typing import Iterable, Tuple, List, Dict, Union

def offdiag_2local_terms(
    n: int,
    triples: Iterable[Union[Tuple[int, int, int], Tuple[int, int, int, float]]],
    *,
    indexing: str = "lsb",
    merge_tol: float = 1e-12,
) -> List[Tuple[complex, str]]:
    """
    Build pauli_terms that realize chosen off-diagonal entries on each pair (i, j),
    allowing an optional amplitude per triple and MERGING identical Pauli strings.

    triples entries:
        (i, j, ℓ)             -> amplitude = +1.0
        (i, j, ℓ, amplitude)  -> use provided (possibly complex) amplitude

    ℓ mapping (same as before):
    
        0:  (0,1) :  1/2 ( I⊗X + Z⊗X ) = P0 ⊗ X
        1:  (2,3) :  1/2 ( I⊗X - Z⊗X ) = P1 ⊗ X
        2:  (0,2) :  1/2 ( X⊗I + X⊗Z ) = X ⊗ P0
        3:  (1,3) :  1/2 ( X⊗I - X⊗Z ) = X ⊗ P1
        4:  (0,3) :  1/2 ( X⊗X - Y⊗Y ) = 00 <-> 11
        5:  (1,2) :  1/2 ( X⊗X + Y⊗Y ) = 01 <-> 10
    """
    acc: Dict[str, complex] = defaultdict(complex)  # pauli_string -> summed coeff
    order: Dict[str, int] = {}                      # first-seen order for stable output
    next_ord = 0

    def _see(p: str):
        nonlocal next_ord
        if p not in order:
            order[p] = next_ord
            next_ord += 1

    # helpers to build pauli strings
    def _with_char(pos: int, ch: str) -> List[str]:
        p = ["I"] * n
        p[_char_pos(pos, n, indexing)] = ch
        return p

    def p_IX(i: int, j: int) -> str:
        p = _with_char(j, "X");              return "".join(p)
    def p_ZX(i: int, j: int) -> str:
        p = _with_char(i, "Z"); p[_char_pos(j, n, indexing)] = "X"; return "".join(p)
    def p_XI(i: int, j: int) -> str:
        p = _with_char(i, "X");              return "".join(p)
    def p_XZ(i: int, j: int) -> str:
        p = _with_char(i, "X"); p[_char_pos(j, n, indexing)] = "Z"; return "".join(p)
    def p_XX(i: int, j: int) -> str:
        p = _with_char(i, "X"); p[_char_pos(j, n, indexing)] = "X"; return "".join(p)
    def p_YY(i: int, j: int) -> str:
        p = _with_char(i, "Y"); p[_char_pos(j, n, indexing)] = "Y"; return "".join(p)

    for t in triples:
        if len(t) == 3:
            i, j, ell = t
            amp = 1.0
        elif len(t) == 4:
            i, j, ell, amp = t
        else:
            raise ValueError(f"Each triple must be (i,j,ℓ) or (i,j,ℓ,amp); got {t}")
        
        if type(ell) == str:
            if ell == "00 01" or ell == "01 00":
                ell = 0
            elif ell == "10 11" or ell == "11 10":
                ell = 1
            elif ell == "00 10" or ell == "10 00":
                ell = 2
            elif ell == "01 11" or ell == "11 01":
                ell = 3
            elif ell == "00 11" or ell == "11 00":
                ell = 4
            elif ell == "01 10" or ell == "10 01":
                ell = 5
            else:
                raise ValueError(f"{ell} is not a valid string")

        if not (0 <= i < n and 0 <= j < n):
            raise ValueError(f"(i,j)=({i},{j}) out of range for n={n}")
        if i == j:
            continue
        if ell not in (0,1,2,3,4,5):
            raise ValueError(f"ℓ must be in 0..5, got {ell}")

        # per-ℓ decomposition (same as before, then scaled by amp)
        if ell == 0:      # 1/2 (I⊗X + Z⊗X)
            terms = [(p_IX(i,j), 0.5), (p_ZX(i,j), 0.5)]
        elif ell == 1:    # 1/2 (I⊗X - Z⊗X)
            terms = [(p_IX(i,j), 0.5), (p_ZX(i,j), -0.5)]
        elif ell == 2:    # 1/2 (X⊗I + X⊗Z)
            terms = [(p_XI(i,j), 0.5), (p_XZ(i,j), 0.5)]
        elif ell == 3:    # 1/2 (X⊗I - X⊗Z)
            terms = [(p_XI(i,j), 0.5), (p_XZ(i,j), -0.5)]
        elif ell == 4:    # 1/2 (XX - YY)
            terms = [(p_XX(i,j), 0.5), (p_YY(i,j), -0.5)]
        else:             # ell == 5: 1/2 (XX + YY)
            terms = [(p_XX(i,j), 0.5), (p_YY(i,j), 0.5)]

        amp = complex(amp)
        for pstr, c in terms:
            _see(pstr)
            acc[pstr] += amp * c

    # produce merged list, dropping tiny coefficients
    merged = [(coef, pstr) for pstr, coef in acc.items() if abs(coef) > merge_tol]
    # stable order by first appearance
    merged.sort(key=lambda x: order[x[1]])
    return merged

def simple_plot(pauli_terms, 
                penalty_pairs=None,
                hamming_k=None,
                partition_weights=None,
                curve_step=0):
    G, legend, get_pos = pauli_cyto_widget(pauli_terms, 
                                        layout_name='cose', 
                                        opt_passes=1, 
                                        hamming_k=hamming_k, 
                                        partition_weights=partition_weights,
                                        
                                        curve_step = curve_step,
                                        show_labels = False,
                                        show_weights = False,
                                        
                                        enforce_bipartite = False,
                                        bipartite_even_rev = False,
                                        bipartite_odd_rev = True,
                                        
                                        input_mode = "pauli",
                                        local_triples = None,
                                        indexing = "msb",
                                        
                                        layout_kwargs=dict(numIter=4000,
                                                                coolingFactor=0.99,
                                                                minTemp=1e-3,
                                                                randomize=False, animate=False,
                                                                nodeRepulsion=400000, idealEdgeLength=60, edgeElasticity=100),

                                        penalty_pairs = penalty_pairs)
    display(legend, G)

# ===================== Example (edit freely) =====================
#n = 6; k = 2; xy_pairs = [(0,1),(1,2),(3,4),(4,5)] #2D 3x3 Lattice
#n = 9; k = 3; xy_pairs = [(0,1),(1,2),(3,4),(4,5),(6,7),(7,8)] #3D 3x3x3 Lattice
#n = 6; k = 3; xy_pairs = [(0,1),(1,2),(3,4),(4,5),(1,4)] #sphere molecule
#n = 8; k = 2; xy_pairs = [(0,1),(1,2),(2,3),(4,5),(5,6),(6,7)] #2D 4x4 lattice
#n = 10; k = 5; xy_pairs = [(0,1),(1,2),(2,3),(3,4),(5,6),(6,7),(7,8),(8,9),(2,7)] #sphere molecule
#n = 10; k = 2; xy_pairs = [(0,1),(0,2),(0,3),(1,4),(1,5),(2,6),(2,7),(3,8),(3,9)]  #ternary-tree qubit layout
#n = 8; k = 4; xy_pairs = [(0,1),(1,2),(2,3),(3,0),(0,4),(1,5),(2,6),(3,7)]  #square-tree qubit layout
#n = 7; k = 3; xy_pairs = [(0,1),(0,2),(1,3),(1,4),(2,5),(2,6)]  #binary-tree qubit layout
#n = 9; k = 2; xy_pairs = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(6,7)] #prism
#n = 4; k = 2; xy_pairs = [(0,1),(0,2),(0,3)] #hexagon (4 qubits)
#n = 8; k = 4; xy_pairs = [(0,1),(0,2),(0,3),(4,5),(4,6),(4,7)] #hexagon (4 qubits)
#n = 7; k = 4; xy_pairs = [(0,1),(0,2),(0,3),(4,5),(5,6)] #hexagon cylindar
#n = 6; k = 2; xy_pairs = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)] #hexagon cylindar
#pauli_terms = xy_pauli_terms(n, xy_pairs)

'''
for i in [2]: #range(6):
    for j in [4]: #range(i, 6):
        for k in [0]: #range(1):
            print("(0, 1):", i, LSB_list[i])
            print("(2, 3):", j, LSB_list[j])
            print("(1, 2):", k, LSB_list[k])
            pauli_terms = offdiag_2local_terms(5, [(0,1,i),(2,3,j),(1,2,k)])

        # Qubit layout
        #G_q, legend, get_pos = pauli_cyto_widget(pauli_terms, layout_name='cose', opt_passes=1, hamming_k=1, partition_weights=None,
        #                                       layout_kwargs=dict(numIter=1000, coolingFactor=0.995))
        #display(legend, G_q)

        # 2D draggable (with global Hamming weight = 3)
        G, legend, get_pos = pauli_cyto_widget(pauli_terms, layout_name='cose', opt_passes=1, hamming_k=None, partition_weights=None,
                                            layout_kwargs=dict(numIter=1000, coolingFactor=0.995))
        display(legend, G)
'''



# Optional: enforce per-partition weights, e.g., split into two halves with (k_left, k_right) = (2,1)
#G_part, legend_part, _ = pauli_cyto_widget(pauli_terms, partition_weights=(2,1))
#display(legend_part, G_part)

# Optional: rotatable 3D view (install once: %pip install plotly)
# Draggable 3D widget:
#fig = pauli_plotly_3d(pauli_terms, hamming_k=k)
#display(fig)

'\nfor i in [2]: #range(6):\n    for j in [4]: #range(i, 6):\n        for k in [0]: #range(1):\n            print("(0, 1):", i, LSB_list[i])\n            print("(2, 3):", j, LSB_list[j])\n            print("(1, 2):", k, LSB_list[k])\n            pauli_terms = offdiag_2local_terms(5, [(0,1,i),(2,3,j),(1,2,k)])\n\n        # Qubit layout\n        #G_q, legend, get_pos = pauli_cyto_widget(pauli_terms, layout_name=\'cose\', opt_passes=1, hamming_k=1, partition_weights=None,\n        #                                       layout_kwargs=dict(numIter=1000, coolingFactor=0.995))\n        #display(legend, G_q)\n\n        # 2D draggable (with global Hamming weight = 3)\n        G, legend, get_pos = pauli_cyto_widget(pauli_terms, layout_name=\'cose\', opt_passes=1, hamming_k=None, partition_weights=None,\n                                            layout_kwargs=dict(numIter=1000, coolingFactor=0.995))\n        display(legend, G)\n'

## XX + YY Pauli generation

In [ ]:
import itertools

try:
    import igraph as ig
except ImportError:
    ig = None


def _is_connected(edges, n):
    """
    Check if a simple graph on n vertices with edge set `edges`
    is connected.
    edges: iterable of (i, j) with 0 <= i < j < n.
    """
    if n <= 1:
        return True

    if not edges:
        return False

    # build adjacency list
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    # DFS/BFS from vertex 0
    seen = [False] * n
    stack = [0]
    seen[0] = True

    while stack:
        u = stack.pop()
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                stack.append(v)

    return all(seen)


def _canonical_code_slow(edges, n):
    """
    Fallback: brute-force canonical code by trying all permutations.
    edges: set of (i, j) with 0 <= i < j < n.
    Returns a lexicographically minimal upper-triangular bitstring.
    """
    edges = {tuple(sorted(e)) for e in edges}
    best = None
    for perm in itertools.permutations(range(n)):
        bits = []
        for i in range(n):
            for j in range(i + 1, n):
                u, v = perm[i], perm[j]
                if (min(u, v), max(u, v)) in edges:
                    bits.append('1')
                else:
                    bits.append('0')
        code = ''.join(bits)
        if best is None or code < best:
            best = code
    return best


def _canonical_code_igraph(edges, n):
    """
    Canonical code using igraph's canonical_permutation (much faster).
    edges: set of (i, j) with 0 <= i < j < n.
    """
    if not edges:
        # canonical representation of empty graph
        return ()

    g = ig.Graph(n=n, edges=list(edges), directed=False)
    perm = g.canonical_permutation()
    # If your igraph returns (perm, colors), use: perm = perm[0]
    canon_edges = sorted(
        (min(perm[u], perm[v]), max(perm[u], perm[v]))
        for (u, v) in edges
    )
    return tuple(canon_edges)


def _canonical_code(edges, n):
    """
    Wrapper: use igraph if available, otherwise fall back to slow brute-force.
    """
    if ig is not None:
        return _canonical_code_igraph(edges, n)
    else:
        return _canonical_code_slow(edges, n)


def generate_xx_yy_hamiltonians_unlabeled(n, connected_only=False):
    """
    Generate all distinct XX+YY Hamiltonians on n qubits, up to qubit relabeling.

    Each Hamiltonian corresponds to an unlabeled simple graph on n vertices.
    For each edge (i, j), we add two Pauli strings:
        (1, string with X at i,j)
        (1, string with Y at i,j)

    Parameters
    ----------
    n : int
        Number of qubits / vertices.
    connected_only : bool, optional
        If True, only include Hamiltonians whose underlying qubit graph
        is connected (i.e. every qubit has a path to every other).

    Returns
    -------
    hams : list[list[tuple[int, str]]]
        List of Hamiltonians. Each Hamiltonian is a list of terms (coef, pauli_string),
        e.g. (1, "IIXXII") for n = 6.
    """
    if n <= 1:
        # Only the empty graph -> empty Hamiltonian
        if connected_only and n == 0:
            return []  # no vertices, define as "no connected graph"
        return [[]]

    # all possible edges in a labeled graph on n vertices
    all_edges = [(i, j) for i in range(n) for j in range(i + 1, n)]
    m = len(all_edges)

    seen_codes = set()
    hams = []

    # connectivity: at least n-1 edges needed
    min_edges = n - 1 if connected_only else 0

    # enumerate graphs by number of edges r
    for r in range(min_edges, m + 1):
        for subset in itertools.combinations(all_edges, r):
            edges = set(subset)

            if connected_only and not _is_connected(edges, n):
                continue

            # canonicalize (mod out by vertex permutations)
            code = _canonical_code(edges, n)
            if code in seen_codes:
                continue
            seen_codes.add(code)

            # convert this edge set into XX+YY Hamiltonian
            terms = []
            for (i, j) in edges:
                # XX term
                p = ['I'] * n
                p[i] = 'X'
                p[j] = 'X'
                terms.append((1, ''.join(p)))

                # YY term
                p = ['I'] * n
                p[i] = 'Y'
                p[j] = 'Y'
                terms.append((1, ''.join(p)))

            hams.append(terms)

    return hams

#generate_xx_yy_hamiltonians_unlabeled(7, connected_only=True)

## Individual graph tests

In [ ]:
pauli_terms = []
#pauli_terms += offdiag_2local_terms(5, [(0,1,0), (1,2,0), (0,3,0), (3,4,0)], indexing = "msb") #4 by 4 lattice
#pauli_terms += offdiag_2local_terms(6, [(0,1,0), (2,3,0), (4,5,0)], indexing = "msb"); pauli_terms += [(1, "XIIIII"), (1, "IIXIII"), (1, "IIIIXI")]
#pauli_terms += [(1, "XI"), (1, "IX"), (1, "XX")]
#pauli_terms += offdiag_2local_terms(2, [(0,1,0, 0.9), (0,1,1, 1), (0,1,5, 1.1)], indexing = "msb")
#pauli_terms += offdiag_2local_terms(2, [(0,1,0, 0.1)], indexing = "msb"); pauli_terms += [(1, "XI")] 

#pauli_terms += offdiag_2local_terms(3, [(0,1,0), (0,2,1)], indexing = "msb"); pauli_terms += [(1, "XII")]

#pauli_terms += offdiag_2local_terms(5, [(0,1,5)], indexing = "msb")
#pauli_terms += offdiag_2local_terms(5, [(0,1,5), (1,2,5), (2,3,5), (3,4,5)], indexing = "msb")
#pauli_terms += offdiag_2local_terms(5, [(2,3,0), (2,3,3)], indexing = "msb")

#pauli_terms += offdiag_2local_terms(5, [(0,4,4), (0,1,2)], indexing = "msb");pauli_terms += [(1, "XIIII")]

#pauli_terms += offdiag_2local_terms(5, [(4,0,0), (4,2,2), (4,1,1), (3,2,3), (3,2,4), (3,2,5)], indexing = "lsb")
#print(pauli_terms)

#pauli_terms += offdiag_2local_terms(5, [(4,0,0), (4,2,2), (3,2,3), (3,2,4)], indexing = "lsb")
#pauli_terms += offdiag_2local_terms(5, [(4,1,1)], indexing = "lsb")
#pauli_terms += offdiag_2local_terms(5, [(3,2,5)], indexing = "lsb")
#pauli_terms += offdiag_2local_terms(4, [(0,1,"11 10"), (0,1,"00 10"), (0,2,"00 10"), (0,2,"00 01")], indexing = "msb")

#pauli_terms += offdiag_2local_terms(3, [(0,1,"00 01"), (0,1,"00 10")], indexing = "msb")

#pauli_terms = [(1, "XXII"), (1, "YYII"), (1, "IXXI"), (1, "IYYI"), (1, "IIXX"), (-1, "IIYY")]
#pauli_terms = [(1, "XXII"), (-1, "YYII"), (1, "XIXI"), (-1, "YIYI"), (1, "XIIX"), (-1, "YIIY")]
#pauli_terms = [(1, "XX"), (1, "YY"), (1, "XI")]

'''
pauli_terms = [(1, "XIII"), (1, "IXII"), (1, "IIXI"), (1, "IIIX")]

#n_override = 3; local_triples = [(1,0,"11 10"), (1,0,"00 10"), (1,2,"00 10"), (1,2,"00 01")] #3-bit chain
for i in range(6):
    print(i)
    n_override = 5; local_triples = [(0, 1, i)] #[(0,1,"00 01"), (0,1,"10 11"), (0,1,"00 11")] + \
                                    #[(1,2,i)]


    G, legend, get_pos = pauli_cyto_widget(None, 
                                        layout_name='cose', 
                                        opt_passes=1, 
                                        hamming_k=None, 
                                        partition_weights=None,
                                        
                                        always_curve = True,
                                        curve_step = -200,
                                        show_labels = True,
                                        show_weights = False,
                                        
                                        enforce_line = True,
                                        
                                        enforce_bipartite = False,
                                        bipartite_group_by_partile_num = True,
                                        bipartite_even_rev = False,
                                        bipartite_odd_rev = False,
                                        
                                        input_mode = "local",
                                        local_triples = local_triples,
                                        n_override = n_override,
                                        indexing = "msb",
                                        
                                        layout_kwargs=dict(numIter=8000,
                                                                coolingFactor=0.95,
                                                                minTemp=1e-3,
                                                                randomize=False, animate=False,
                                                                nodeRepulsion=400000, idealEdgeLength=60, edgeElasticity=100))

    display(legend, G)
'''

'\npauli_terms = [(1, "XIII"), (1, "IXII"), (1, "IIXI"), (1, "IIIX")]\n\n#n_override = 3; local_triples = [(1,0,"11 10"), (1,0,"00 10"), (1,2,"00 10"), (1,2,"00 01")] #3-bit chain\nfor i in range(6):\n    print(i)\n    n_override = 5; local_triples = [(0, 1, i)] #[(0,1,"00 01"), (0,1,"10 11"), (0,1,"00 11")] +                                     #[(1,2,i)]\n\n\n    G, legend, get_pos = pauli_cyto_widget(None, \n                                        layout_name=\'cose\', \n                                        opt_passes=1, \n                                        hamming_k=None, \n                                        partition_weights=None,\n                                        \n                                        always_curve = True,\n                                        curve_step = -200,\n                                        show_labels = True,\n                                        show_weights = False,\n                                        \n            

## Plot pauli terms

In [ ]:
#pauli_terms += offdiag_2local_terms(3, [(0,1,"11 10"), (0,1,"00 10"), (0,2,"00 10"), (0,2,"00 01")], indexing = "msb")
#pauli_terms = [(1, "XXIII"), (1, "YYIII"), (1, "IXXII"), (1, "IYYII"), (1, "IIXXI"), (1, "IIYYI"), (1, "IIIXX"), (1, "IIIYY")]
#pauli_terms = [(1, "XXIII"), (1, "IXXII"), (1, "IIXXI"), (1, "IIIXX")]

'''
for pauli_terms in [#[(1, "IXIII"), (1, "IIXII"), (1, "IIIXI"), (1, "IIIIX")],
                    #[(1, "XX")],
                    #[(1, "XXI")],
                    #[(1, "XXI"), (1, "IXX")],
                    #[(1, "XXII"), (1, "IXXI")],
                    #[(1, "XXII"), (1, "IXXI"), (1, "IIXX")],
                    #[(1, "XXIII"), (1, "XIXII"), (1, "XIIXI"), (1, "XIIIX")]]:
                    #[(1, "XXIIII"), (1, "YYIIII"), (1, "IXXIII"), (1, "IYYIII"), (1, "IIXXII"), (1, "IIYYII"), (1, "IIIXXI"), (1, "IIIYYI"), (1, "IIIIXX"), (1, "IIIIYY"), (1, "XIIIIX"), (-1, "YIIIIY"), (1, "IIIIIX"), (1, "IIIZIX")]]:
                    #[(1, "IIX"), (1, "IXI"), (1, "XII")],
                    #[(1, "IIX"), (1, "IXI"), (1, "XII"), (1, "ZIX"), (-1, "ZXI")]]:
                    #[(1, "XYII"), (1, "YXII"),(1, "IXYI"), (1, "IYXI"),(1, "IIXY"), (1, "IIYX")]]:
                    #[(1, "XXII"), (1, "IXXI"), (1, "IIXX"), (1, "IIIX"), (1, "IIXI"), (1, "IXII"), (1, "XIII"),]]:
                    #[(1, "XXII"), (1, "YYII"), (1, "IXXI"), (1, "IYYI"), (1, "IIXX"), (1, "IIYY"), (1, "XIIX"), (1, "YIIY")],
                    #[(1, "XXII"), (-1, "YYII"), (1, "IXXI"), (-1, "IYYI"), (1, "IIXX"), (-1, "IIYY"), (1, "XIIX"), (-1, "YIIY")],
                    #[(1, "XXII"), (1, "IXXI"), (1, "IIXX"), (1, "XIIX")]]:
                    [(1, "XXIIII")]]:
                    
    G, legend, get_pos = pauli_cyto_widget(pauli_terms, 
                                        layout_name='cose', 
                                        opt_passes=1, 
                                        hamming_k=None, 
                                        partition_weights=None,
                                        
                                        curve_step = 0,
                                        show_labels = False,
                                        show_weights = False,
                                        
                                        enforce_bipartite = False,
                                        bipartite_even_rev = False,
                                        bipartite_odd_rev = True,
                                        
                                        input_mode = "pauli",
                                        local_triples = None,
                                        indexing = "msb",
                                        
                                        layout_kwargs=dict(numIter=8000,
                                                                coolingFactor=0.99,
                                                                minTemp=1e-3,
                                                                randomize=False, animate=False,
                                                                nodeRepulsion=400000, idealEdgeLength=60, edgeElasticity=100))

    display(legend, G)
'''

'\nfor pauli_terms in [#[(1, "IXIII"), (1, "IIXII"), (1, "IIIXI"), (1, "IIIIX")],\n                    #[(1, "XX")],\n                    #[(1, "XXI")],\n                    #[(1, "XXI"), (1, "IXX")],\n                    #[(1, "XXII"), (1, "IXXI")],\n                    #[(1, "XXII"), (1, "IXXI"), (1, "IIXX")],\n                    #[(1, "XXIII"), (1, "XIXII"), (1, "XIIXI"), (1, "XIIIX")]]:\n                    #[(1, "XXIIII"), (1, "YYIIII"), (1, "IXXIII"), (1, "IYYIII"), (1, "IIXXII"), (1, "IIYYII"), (1, "IIIXXI"), (1, "IIIYYI"), (1, "IIIIXX"), (1, "IIIIYY"), (1, "XIIIIX"), (-1, "YIIIIY"), (1, "IIIIIX"), (1, "IIIZIX")]]:\n                    #[(1, "IIX"), (1, "IXI"), (1, "XII")],\n                    #[(1, "IIX"), (1, "IXI"), (1, "XII"), (1, "ZIX"), (-1, "ZXI")]]:\n                    #[(1, "XYII"), (1, "YXII"),(1, "IXYI"), (1, "IYXI"),(1, "IIXY"), (1, "IIYX")]]:\n                    #[(1, "XXII"), (1, "IXXI"), (1, "IIXX"), (1, "IIIX"), (1, "IIXI"), (1, "IXII"), (1, "XIII"),]]:\n    

In [ ]:
#save_parts(G, "individual_graphs/n7_256_3D_cube.json.gz")

# Static Pauli plot

In [61]:
import math
from collections import defaultdict
from typing import List, Tuple, Optional, Sequence, Dict, Any

import numpy as np
import matplotlib.pyplot as plt


def _rescale_positions_visible(pos: Dict[int, Tuple[float, float]],
                               visible: Sequence[int],
                               pad: float = 0.08):
    """
    Affine-rescale positions of VISIBLE nodes to [pad, 1-pad]^2.
    Keeps aspect ratio. Other nodes in pos are left unchanged.
    """
    vis = list(visible)
    if not vis:
        return pos

    xs = [pos[u][0] for u in vis]
    ys = [pos[u][1] for u in vis]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    dx = max(xmax - xmin, 1e-9)
    dy = max(ymax - ymin, 1e-9)

    scale = (1 - 2 * pad) / max(dx, dy)

    cx = 0.5 * (xmin + xmax)
    cy = 0.5 * (ymin + ymax)

    pos2 = dict(pos)
    for u in vis:
        x, y = pos[u]
        x0 = (x - cx) * scale + 0.5
        y0 = (y - cy) * scale + 0.5
        pos2[u] = (x0, y0)
    return pos2


def _build_pauli_graph_data_static(
    pauli_terms: List[Tuple[complex, str]],
    cancel_tol: float = 1e-12,
    keep_zero_pairs: bool = False,
    per_label_tol: float = 1e-12,
    hamming_k: Optional[int] = None,
    partition_weights: Optional[Sequence[int]] = None,
    penalty_pairs: Optional[List[Tuple[int, int, str]]] = None,
    r_step: float = 120.0,
    width_by_ampl: bool = True,
    base_width: float = 2.0,
    width_from_weight: bool = False,
    width_ref: float = 1.0,
):
    """
    Build data for static drawing, using the same layered layout as cyto.

    - Nodes: bitstrings
    - Edges: colored by Pauli labels, no text on edges
    """
    # survivors: {(u,v): {pauli_label: coeff_eff}}
    survivors, n = build_pairs_by_label(
        pauli_terms,
        cancel_tol=cancel_tol,
        keep_zero_pairs=keep_zero_pairs,
        per_label_tol=per_label_tol,
    )

    N = 1 << n

    # visible nodes
    visible = set(range(N)) if hamming_k is None else {
        u for u in range(N) if _popcount(u) == hamming_k
    }
    if partition_weights is not None:
        include = partition_weight_filter(n, partition_weights)
        visible &= {u for u in range(N) if include(u)}

    # penalties (same convention as your cyto code)
    if penalty_pairs is not None:
        for i, j, pstr in penalty_pairs:
            forbidden_patterns = set(pstr.split())
            to_remove = set()
            for idx in visible:
                b_i = (idx >> (n - i - 1)) & 1
                b_j = (idx >> (n - j - 1)) & 1
                pattern = f"{b_i}{b_j}"
                if pattern in forbidden_patterns:
                    to_remove.add(idx)
            visible -= to_remove

    # base layout: same layered layout as in cyto
    pos = _layered_positions(n, r_step=r_step)

    # rescale visible nodes into [0,1] box with padding
    pos = _rescale_positions_visible(pos, visible, pad=0.12)

    # palette and ordering
    color_for, label_order, label_to_index = palette_from_input_order(pauli_terms)

    # widths by amplitude
    amps = [abs(z) for tm in survivors.values() for z in tm.values()]
    max_amp = max(amps) if amps else 1.0

    edges = []
    labels_present = set()

    for (u, v), termmap in survivors.items():
        if (u not in visible) or (v not in visible):
            continue
        items = sorted(termmap.items(), key=lambda kv: label_to_index[kv[0]])
        for lab, coeff_eff in items:
            mag = abs(complex(coeff_eff))
            width = base_width * (0.5 + 0.5 * (mag / max_amp)) if width_by_ampl else base_width
            if width_from_weight:
                scale = mag / max(1e-12, float(width_ref))
                width = min(10.0, max(0.5, width * scale))

            edges.append({
                "u": u,
                "v": v,
                "lab": lab,
                "color": color_for(lab),
                "width": float(width),
            })
            labels_present.add(lab)

    nodes = []
    for u in range(N):
        if u not in visible:
            continue
        x, y = pos[u]
        nodes.append({
            "idx": u,
            "x": float(x),
            "y": float(y),
            "label": _bitstring(u, n),
            "hamming": _popcount(u),
        })

    # legend entries ordered like cyto
    legend_labels = [lab for lab in label_order if lab in labels_present]
    legend = [(lab, color_for(lab)) for lab in legend_labels]

    return {
        "n": n,
        "N": N,
        "nodes": nodes,
        "edges": edges,
        "pos": pos,
        "legend": legend,
    }


def _draw_pauli_graph_on_ax_no_edge_labels(
    ax,
    graph_data: Dict[str, Any],
    node_size: float = 22.0,
    node_fontsize: int = 10,
    legend_fontsize: int = 9,
):
    """
    Draw Pauli graph:

    - bitstrings above nodes
    - colored edges, no text on edges
    - Pauli legend at top
    """
    nodes = graph_data["nodes"]
    edges = graph_data["edges"]
    pos = graph_data["pos"]
    legend = graph_data["legend"]

    xs = [nd["x"] for nd in nodes]
    ys = [nd["y"] for nd in nodes]
    if not xs:
        ax.axis("off")
        return

    ax.scatter(xs, ys, s=node_size**2, c="#111111", zorder=2)

    # vertical offset for bitstring labels
    y_min, y_max = min(ys), max(ys)
    y_span = max(y_max - y_min, 1e-6)
    y_offset = 0.05 * y_span

    for nd in nodes:
        ax.text(
            nd["x"], nd["y"] + y_offset,
            nd["label"],
            ha="center", va="bottom",
            fontsize=node_fontsize,
            zorder=3,
        )

    for e in edges:
        u, v = e["u"], e["v"]
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        ax.plot(
            [x1, x2], [y1, y2],
            color=e["color"],
            linewidth=e["width"],
            solid_capstyle="round",
            alpha=0.95,
            zorder=1,
        )

    # Legend (Pauli labels) in axes coords
    if legend:
        x0 = 0.02
        dx = 0.18
        y0 = 1.03
        for i, (lab, color) in enumerate(legend):
            x = x0 + i * dx
            ax.text(
                x, y0,
                "■",
                transform=ax.transAxes,
                ha="center", va="center",
                fontsize=legend_fontsize + 2,
                color=color,
                zorder=4,
            )
            ax.text(
                x, y0 + 0.07,
                lab,
                transform=ax.transAxes,
                ha="center", va="bottom",
                fontsize=legend_fontsize,
                color="black",
                zorder=4,
            )

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.18)
    ax.set_aspect("equal")
    ax.axis("off")


def pauli_static_grid(
    pauli_terms_list: Sequence[List[Tuple[complex, str]]],
    ncols: int = 3,
    titles: Optional[Sequence[str]] = None,
    figsize: Optional[Tuple[float, float]] = None,
    # graph-building options
    cancel_tol: float = 1e-12,
    keep_zero_pairs: bool = False,
    per_label_tol: float = 1e-12,
    hamming_k: Optional[int] = None,
    partition_weights: Optional[Sequence[int]] = None,
    penalty_pairs: Optional[List[Tuple[int, int, str]]] = None,
    r_step: float = 120.0,
    width_by_ampl: bool = True,
    base_width: float = 2.0,
    width_from_weight: bool = False,
    width_ref: float = 1.0,
    # drawing options
    node_size: float = 22.0,
    node_fontsize: int = 10,
    legend_fontsize: int = 9,
):
    """
    Grid of static Pauli graphs, layout matching cyto's layered layout
    (no spring-relax).

    - Edges colored, no labels
    - Bitstrings above nodes
    - Pauli legend at top of each subplot
    """
    m = len(pauli_terms_list)
    if titles is not None and len(titles) != m:
        raise ValueError("len(titles) must match len(pauli_terms_list)")

    ncols = max(1, int(ncols))
    nrows = math.ceil(m / ncols)

    if figsize is None:
        figsize = (4.3 * ncols, 4.0 * nrows)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = np.array([axes])
    elif ncols == 1:
        axes = np.array([[ax] for ax in axes])

    axes_flat = axes.flatten()

    for i, terms in enumerate(pauli_terms_list):
        ax = axes_flat[i]
        gdata = _build_pauli_graph_data_static(
            terms,
            cancel_tol=cancel_tol,
            keep_zero_pairs=keep_zero_pairs,
            per_label_tol=per_label_tol,
            hamming_k=hamming_k,
            partition_weights=partition_weights,
            penalty_pairs=penalty_pairs,
            r_step=r_step,
            width_by_ampl=width_by_ampl,
            base_width=base_width,
            width_from_weight=width_from_weight,
            width_ref=width_ref,
        )
        _draw_pauli_graph_on_ax_no_edge_labels(
            ax,
            gdata,
            node_size=node_size,
            node_fontsize=node_fontsize,
            legend_fontsize=legend_fontsize,
        )
        if titles is not None:
            ax.set_title(titles[i], fontsize=12, pad=18)

    # hide unused axes
    for j in range(m, len(axes_flat)):
        axes_flat[j].axis("off")

    fig.subplots_adjust(top=0.9, wspace=0.3, hspace=0.4)
    return fig, axes

# XX+YY tests

## Individual test

In [21]:
'''
# 8-qubit XX+YY with single X0
pauli_terms = [(1, "XXIIIIII"), (1, "YYIIIIII"),
               (1, "IXXIIIII"), (1, "IYYIIIII"),
               (1, "IIXXIIII"), (1, "IIYYIIII"),
               (1, "IIIXXIII"), (1, "IIIYYIII"),
               (1, "IIIIXXII"), (1, "IIIIYYII"),
               (1, "IIIIIXXI"), (1, "IIIIIYYI"),
               (1, "IIIIIIXX"), (1, "IIIIIIYY"),
               (1, "XIIIIIII")]
'''

'''
pauli_terms = [(1, "XXIIIIIII"), (1, "YYIIIIIII"),
               (1, "IXXIIIIII"), (1, "IYYIIIIII"),
               (1, "IIXXIIIII"), (1, "IIYYIIIII"),
               (1, "IIIXXIIII"), (1, "IIIYYIIII"),
               (1, "IIIIXXIII"), (1, "IIIIYYIII"),
               (1, "IIIIIXXII"), (1, "IIIIIYYII"),
               (1, "IIIIIIXXI"), (1, "IIIIIIYYI"),
               (1, "IIIIIIIXX"), (1, "IIIIIIIYY"),
               (1, "XIIIIIIII")]
'''

'''
# IX+ZX with single X0
pauli_terms = [(1, "XIIIIII"), 
               (1, "IXIIIII"), (1, "ZXIIIII"),
               (1, "IIXIIII"), (1, "IZXIIII"),
               (1, "IIIXIII"), (1, "IIZXIII"),
               (1, "IIIIXII"), (1, "IIIZXII"),
               (1, "IIIIIXI"), (1, "IIIIZXI"),
               (1, "IIIIIIX"), (1, "IIIIIZX")]
'''

'''
# IX+ZX mixed with IX-Zx with single X0
pauli_terms = [#(1, "XIIII"), 
               (1, "IXIIIII"), (1, "ZXIIIII"),
               (1, "XIIIIII"), (1, "XZIIIII"),
               (1, "IIXIIII"), (1, "IZXIIII"),
               (1, "IXIIIII"), (1, "IXZIIII"),
               (1, "IIIXIII"), (1, "IIZXIII"),
               (1, "IIXIIII"), (1, "IIXZIII"),
               (1, "IIIIXII"), (1, "IIIZXII"),
               (1, "IIIXIII"), (1, "IIIXZII"),
               (1, "IIIIIXI"), (1, "IIIIZXI"),
               (1, "IIIIXII"), (1, "IIIIXZI"),
               (1, "IIIIIIX"), (1, "IIIIIZX"),
               (1, "IIIIIXI"), (1, "IIIIIXZ")]
               #(1, "IIIIIXI"), (1, "IIIIZXI"),
               #(1, "IIIIIIX"), (-1, "IIIIIZX")]
'''

# XX Tree
#pauli_terms = [(1, "XXIII"), (1, "XIXII"), (1, "XIIXI"), (1, "XIIIX"), (1, "IXXII")]
#pauli_terms = [(1, "XXI"), (1, "IXX"), (1, "XIX")]
#pauli_terms = [(1, "XII"), (1, "IXI"), (1, "IIX"), (1, "XXX")]
#pauli_terms = [(1, "XXII"), (1, "IXXI"), (1, "IIXX"), (1, "XIIX")]

'''
# 5 qubit chain XX+YY
pauli_terms = [(1, "XXIIIII"), (1, "YYIIIII"),
               (1, "IXXIIII"), (1, "IYYIIII"),
               (1, "IIXXIII"), (1, "IIYYIII"),
               (1, "IIIXXII"), (1, "IIIYYII"),
               (1, "IIIIXXI"), (1, "IIIIYYI"),
               (1, "IIIIIXX"), (1, "IIIIIYY"),]
'''

'''
pauli_terms = [(1, "XXIII"), (1, "YYIII"),
               (1, "IXIXI"), (1, "IYIYI"),
               (1, "IIXXI"), (1, "IIYYI"),
               (1, "IIIXX"), (1, "IIIYY"),
               (1, "XIIIX"), (1, "YIIIY")]
'''


'''
pauli_terms = [(1, 'XXIIII'), (1, 'YYIIII'), 
            (1, 'IXXIII'), (1, 'IYYIII'), 
            (1, 'XIIXII'), (1, 'YIIYII'), 
            (1, 'IXIIXI'), (1, 'IYIIYI'), 
            (1, 'XIXIII'), (1, 'YIYIII'), 
            (1, 'IIXIIX'), (1, 'IIYIIY')]
'''

'''
# S^+_0 S^+_1 S^-_2 S^-_3 + S^+_3 S^+_2 S^-_1 S^-_0 
pauli_terms = [
    ( 0.125, "XXXXII"),
    (-0.125, "XXYYII"),
    ( 0.125, "XYXYII"),
    ( 0.125, "XYYXII"),
    ( 0.125, "YXXYII"),
    ( 0.125, "YXYXII"),
    (-0.125, "YYXXII"),
    ( 0.125, "YYYYII"),
]
'''

'''
# S^+_0 S^+_1 S^-_1 S^-_2 + S^+_2 S^+_1 S^-_1 S^-_0 
pauli_terms = [
    ( 0.25, "XIXIII"),   # X0 I1 X2 I3 I4 I5
    ( 0.25, "YIYIII"),   # Y0 I1 Y2 I3 I4 I5
    (-0.25, "XZXIII"),   # X0 Z1 X2 I3 I4 I5
    (-0.25, "YZYIII"),   # Y0 Z1 Y2 I3 I4 I5
]
'''

'''
# pin one particle, move the other
pauli_terms = [
    (1, "IXXIII"), (-1, "ZXXIII"),
    (1, "IYYIII"), (-1, "ZYYIII"),  
    
    (1, "IIXXII"), (-1, "ZIXXII"),  
    (1, "IIYYII"), (-1, "ZIYYII"),  
    
    (1, "IIIXXI"), (-1, "ZIIXXI"),
    (1, "IIIYYI"), (-1, "ZIIYYI"),  
    
    (1, "IIIIXX"), (-1, "ZIIIXX"),
    (1, "IIIIYY"), (-1, "ZIIIYY"),
      
    (1, "XXIIII"), (-1, "XXIIIZ"),  
    (1, "YYIIII"), (-1, "YYIIIZ"),  
    
    (1, "IXXIII"), (-1, "IXXIIZ"),  
    (1, "IYYIII"), (-1, "IYYIIZ"),  
    
    (1, "IIXXII"), (-1, "IIXXIZ"),  
    (1, "IIYYII"), (-1, "IIYYIZ"), 
    
    (1, "IIIXXI"), (-1, "IIIXXZ"),  
    (1, "IIIYYI"), (-1, "IIIYYZ"), 
]
'''

'''
# 6 qubit XX+YY 2-particle space cut into line
pauli_terms = [(1, "XXIIII"),(1, "YYIIII"),
               (1, "IXXIII"),(1, "IYYIII"),
               (1, "IIXXII"),(1, "IIYYII"),
               (1, "IIIXXI"),(1, "IIIYYI"),
               (1, "IIIIXX"),(1, "IIIIYY"),
               
               (0.5, "IIIXIX"),(0.5, "IIIYIY"),
               (-0.5, "IIIXZX"),(-0.5, "IIIYZY"),
               
               (0.5, "XIXIII"),(0.5, "YIYIII"),
               (-0.5, "XZXIII"),(-0.5, "YZYIII"),
               
               (-0.5, "IIXXII"),(-0.5, "IIYYII"),
               (0.5, "IZXXII"),(0.5, "IZYYII"),
               
               (-0.5, "IIXXII"),(-0.5, "IIYYII"),
               (0.5, "IIXXZI"),(0.5, "IIYYZI"),
               
               (-0.5, "XXIIII"),(-0.5, "YYIIII"),
               (0.5, "XXZIII"),(0.5, "YYZIII"),
               
               (-0.5, "XXIIII"),(-0.5, "YYIIII"),
               (0.5, "XXIZII"),(0.5, "YYIZII"),
               
               (-0.5, "XXIIII"),(-0.5, "YYIIII"),
               (0.5, "XXIIZI"),(0.5, "YYIIZI"),
               
               (-0.5, "IXXIII"),(-0.5, "IYYIII"),
               (0.5, "IXXIZI"),(0.5, "IYYIZI"),
               
               (-0.5, "IXXIII"),(-0.5, "IYYIII"),
               (0.5, "IXXIIZ"),(0.5, "IYYIIZ"),
               
               (-0.5, "IIIXXI"),(-0.5, "IIIYYI"),
               (0.5, "IIIXXZ"),(0.5, "IIIYYZ"),
               
               ]
'''

'''
# 6 qubit tree
pauli_terms = [(1, "XXIIII"),(1, "YYIIII"),
               (1, "XIXIII"),(1, "YIYIII"),
               (1, "XIIXII"),(1, "YIIYII"),
               (1, "XIIIXI"),(1, "YIIIYI"),
               (1, "XIIIIX"),(1, "YIIIIY")]
'''

'''
pauli_terms = [(0.5000000000000001, 'IIXXI'), (0.5000000000000001, 'IIYYI'), 
               (0.5000000000000001, 'IIIXX'), (0.5000000000000001, 'IIIYY'), 
               (0.25000000000000006, 'IXXII'), (0.25000000000000006, 'IYYII'), 
               (-0.25000000000000006, 'ZXXII'), (-0.25000000000000006, 'ZYYII'), 
               (0.125, 'XXXIX'), (-0.125, 'XYYIX'), 
               (0.125, 'XXYIY'), (0.125, 'XYXIY'), 
               (0.125, 'YXYIX'), (0.125, 'YYXIX'), 
               (-0.125, 'YXXIY'), (0.125, 'YYYIY'), 
               (0.125, 'IXXXX'), (-0.125, 'IXYYX'), 
               (0.125, 'IXXYY'), (0.125, 'IXYXY'), 
               (0.125, 'IYXYX'), (0.125, 'IYYXX'), 
               (-0.125, 'IYXXY'), (0.125, 'IYYYY')]
'''


'''
(XX + YY) \otimes (XX + YY) \neq moving two indis particles together
pauli_terms = [(1, "XXXX"),(1, "YXYX"),
               (1, "XYXY"),(1, "YYYY")]
'''

'''
pauli_terms = [(1, "IXXIII"),(1, "IYYIII"),
               (-0.5, "ZXXIII"),(-0.5, "ZYYIII"),
              
               (1, "IIXXII"), (1, "IIYYII"),
               (-0.5, "ZIXXII"), (-0.5, "ZIYYII"),
               (-0.5, "IZXXII"), (-0.5, "IZYYII"),
               
               (1, "IIIXXI"), (1, "IIIYYI"),
               (0.5, "IIIXXZ"), (0.5, "IIIYYZ"),
               
               (1, "IIIIXX"), (1, "IIIIYY"),
               
               (0.5, "XIXIII"), (0.5, "YIYIII"),
               (-0.5, "XZXIII"), (-0.5, "YZYIII"),
               
               #(0.5, "IXXIII"), (0.5, "IYYIII"),
               (-0.5, "IXXIIZ"), (-0.5, "IYYIIZ"),
               
               (0.5, "IIXIXI"), (0.5, "IIYIYI"),
               (-0.5, "IIXZXI"), (-0.5, "IIYZYI"),
               
               #(0.5, "IIIXXI"), (0.5, "IIIYYI"),
               (-0.5, "IIIXXZ"), (-0.5, "IIIYYZ")]
'''

pauli_terms = [(0.5, "XXIIIIII"), (0.5, "YYIIIIII"),
               (0.5, "IXXIIIII"), (0.5, "IYYIIIII"),
               (0.5, "IIXXIIII"), (0.5, "IIYYIIII"),
               
               (-0.25, "IIXXIIII"), (-0.25, "IIYYIIII"),
               (0.25, "ZIXXIIII"), (0.25, "ZIYYIIII"),
               
               (-0.25, "IIXXIIII"), (-0.25, "IIYYIIII"),
               (0.25, "IZXXIIII"), (0.25, "IZYYIIII"),
               
                (0.25, "IXIXIIII"),  (0.25, "IYIYIIII"),
               (-0.25, "IXZXIIII"), (-0.25, "IYZYIIII"),]

G, legend, get_pos = pauli_cyto_widget(pauli_terms, 
                                    layout_name='cose', 
                                    opt_passes=1, 
                                    hamming_k=2,
                                    partition_weights=None,
                                    
                                    curve_step = 0,
                                    show_labels = False,
                                    show_weights = False,
                                    
                                    enforce_bipartite = False,
                                    bipartite_even_rev = False,
                                    bipartite_odd_rev = True,
                                    
                                    input_mode = "pauli",
                                    local_triples = None,
                                    indexing = "msb",
                                    
                                    layout_kwargs=dict(numIter=8000,
                                                            coolingFactor=0.999,
                                                            minTemp=1e-3,
                                                            randomize=False, animate=False,
                                                            nodeRepulsion=400000, idealEdgeLength=60, edgeElasticity=100))

display(legend, G)

n = len(pauli_terms[0][1])
for (u, v), val in pauli_terms_to_basis_elements(n, pauli_terms).items():
    bu, bv = _bitstring(u, n, 'msb'), _bitstring(v, n, 'msb')
    if bu.count("1") == 2:
        print(f"({bu}, {bv})", val)


'''

XY_list = generate_xx_yy_hamiltonians_unlabeled(4, connected_only=True)
fig, axes = pauli_static_grid(
    XY_list,
    ncols=1,
    
    hamming_k=2,
    partition_weights=None,
)

plt.show()
'''

HTML(value="<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'><b>Legend (Paul…

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'randomize': False, 'animate': False, 'numIter': 8000, 'cool…

(10100000, 01100000) (1+0j)
(01100000, 10100000) (1+0j)
(10010000, 01010000) (1+0j)
(01010000, 10010000) (1+0j)
(10001000, 01001000) (1+0j)
(01001000, 10001000) (1+0j)
(10000100, 01000100) (1+0j)
(01000100, 10000100) (1+0j)
(10000010, 01000010) (1+0j)
(01000010, 10000010) (1+0j)
(10000001, 01000001) (1+0j)
(01000001, 10000001) (1+0j)
(11000000, 10100000) (1+0j)
(10100000, 11000000) (1+0j)
(01010000, 00110000) (1+0j)
(00110000, 01010000) (1+0j)
(01001000, 00101000) (1+0j)
(00101000, 01001000) (1+0j)
(01000100, 00100100) (1+0j)
(00100100, 01000100) (1+0j)
(01000010, 00100010) (1+0j)
(00100010, 01000010) (1+0j)
(01000001, 00100001) (1+0j)
(00100001, 01000001) (1+0j)
(00101000, 00011000) (1+0j)
(00011000, 00101000) (1+0j)
(00100100, 00010100) (1+0j)
(00010100, 00100100) (1+0j)
(00100010, 00010010) (1+0j)
(00010010, 00100010) (1+0j)
(00100001, 00010001) (1+0j)
(00010001, 00100001) (1+0j)
(01100000, 00110000) (1+0j)
(00110000, 01100000) (1+0j)


'\n\nXY_list = generate_xx_yy_hamiltonians_unlabeled(4, connected_only=True)\nfig, axes = pauli_static_grid(\n    XY_list,\n    ncols=1,\n    \n    hamming_k=2,\n    partition_weights=None,\n)\n\nplt.show()\n'

## Enumerate all XX+YY 

In [9]:
XY_list =  generate_xx_yy_hamiltonians_unlabeled(5, connected_only=True)#[:200]

In [10]:
for pauli_terms in XY_list:
    print(pauli_terms)
    G, legend, get_pos = pauli_cyto_widget(pauli_terms, 
                                        layout_name='cose', 
                                        opt_passes=1, 
                                        hamming_k=2,
                                        partition_weights=None,
                                        
                                        curve_step = 0,
                                        show_labels = False,
                                        show_weights = False,
                                        
                                        enforce_bipartite = False,
                                        bipartite_even_rev = False,
                                        bipartite_odd_rev = True,
                                        
                                        input_mode = "pauli",
                                        local_triples = None,
                                        indexing = "msb",
                                        
                                        layout_kwargs=dict(numIter=8000,
                                                                coolingFactor=0.99,
                                                                minTemp=1e-3,
                                                                randomize=False, animate=False,
                                                                nodeRepulsion=400000, idealEdgeLength=60, edgeElasticity=100))

    display(legend, G)

[(1, 'XXIII'), (1, 'YYIII'), (1, 'XIXII'), (1, 'YIYII'), (1, 'XIIXI'), (1, 'YIIYI'), (1, 'XIIIX'), (1, 'YIIIY')]


HTML(value="<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'><b>Legend (Paul…

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'randomize': False, 'animate': False, 'numIter': 8000, 'cool…

[(1, 'XXIII'), (1, 'YYIII'), (1, 'XIXII'), (1, 'YIYII'), (1, 'XIIXI'), (1, 'YIIYI'), (1, 'IXIIX'), (1, 'IYIIY')]


HTML(value="<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'><b>Legend (Paul…

...

HTML(value="<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'><b>Legend (Paul…

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'randomize': False, 'animate': False, 'numIter': 8000, 'cool…

[(1, 'XXIII'), (1, 'YYIII'), (1, 'IIXIX'), (1, 'IIYIY'), (1, 'IXXII'), (1, 'IYYII'), (1, 'XIIIX'), (1, 'YIIIY'), (1, 'IIIXX'), (1, 'IIIYY'), (1, 'XIIXI'), (1, 'YIIYI'), (1, 'IXIIX'), (1, 'IYIIY'), (1, 'IIXXI'), (1, 'IIYYI'), (1, 'XIXII'), (1, 'YIYII'), (1, 'IXIXI'), (1, 'IYIYI')]


HTML(value="<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'><b>Legend (Paul…

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'randomize': False, 'animate': False, 'numIter': 8000, 'cool…

## n-particle linear

In [ ]:
from itertools import product
from collections import defaultdict

def pauli_list_to_dict(pauli_list):
    pauli_dict = defaultdict(float)
    for coeff, pauli in pauli_list:
        pauli_dict[pauli] += coeff
    return pauli_dict

def pauli_dict_to_list(pauli_dict):
    pauli_list = []
    for pauli, coeff in pauli_dict.items():
        pauli_list.append((coeff, pauli))
    return pauli_list

def clean_pauli_list(pauli_list):
    return pauli_dict_to_list(pauli_list_to_dict(pauli_list))


def _XX_ij_string(q, i, j):
        s = ["I"]*q
        s[i] = "X"
        s[j] = "X"
        return "".join(s)
    
def _YY_ij_string(q, i, j):
    s = ["I"]*q
    s[i] = "Y"
    s[j] = "Y"
    return "".join(s)

def _XY_ij_string(q, i, j, coeff):
    sx = ["I"]*q
    sx[i] = "X"
    sx[j] = "X"
    
    sy = ["I"]*q
    sy[i] = "Y"
    sy[j] = "Y"
    return [(coeff/2, "".join(sx)), (coeff/2, "".join(sy))] 

'''
def _XX_multiZ_ij_string(i, j, k_list):
    s = ["I"]*q
    s[i] = "X"
    s[j] = "X"
    for k in k_list:
        s[k] = "Z"
    return "".join(s)

def _YY_multiZ_ij_string(i, j, k_list):
    s = ["I"]*q
    s[i] = "Y"
    s[j] = "Y"
    for k in k_list:
        s[k] = "Z"
    return "".join(s)
'''

def _XY_multiP1_ij_string(q, i, j, k_list, coeff):
    
    outputs = []
    klen = len(k_list)
    
    for IZ_str in product(["I", "Z"], repeat=klen):
        sx = ["I"]*q
        sy = ["I"]*q 
        for l, sub in enumerate(IZ_str):
            sx[k_list[l]] = sub
            sy[k_list[l]] = sub
        sx[i] = "X"
        sx[j] = "X"
        sy[i] = "Y"
        sy[j] = "Y"
        sgn = (-1)**(IZ_str.count("Z"))
        outputs.append((sgn*coeff/2**(klen+1), "".join(sx)))
        outputs.append((sgn*coeff/2**(klen+1), "".join(sy)))
    
    return outputs
    

def n_par_linear_scaling(m, n):
    
    q = m * n
    
    pauli_terms = []
    for i in range(m-1):
        pauli_terms += _XY_ij_string(q, i, i+1, 0.5)
    
    if m % 2 == 1:
        for l in range(1, n):
            for i in range(m-1):
                if i % 2 == 1:
                    k_list = [m*lp for lp in range(l)] 
                    pauli_terms += _XY_multiP1_ij_string(q, m*l+i, m*l+i+1, k_list, 1)
                elif i % 2 == 0:
                    k_list = [m*lp-1 for lp in range(1, l+1)]
                    pauli_terms += _XY_multiP1_ij_string(q, m*l+i, m*l+i+1, k_list, 1)
    
    
    elif m % 2 == 0:
        for l in range(1, n):
            for i in range(m-1):
                if i % 2 == 1:
                    k_list = [m*lp for lp in range(l)] 
                    pauli_terms += _XY_multiP1_ij_string(q, m*l+i, m*l+i+1, k_list, 1)
                elif i % 2 == 0:
                    k_list = [m*lp for lp in range(l-1)] + [m*l - 1]
                    pauli_terms += _XY_multiP1_ij_string(q, m*l+i, m*l+i+1, k_list, 1)
            
    return clean_pauli_list(pauli_terms)

def gate_count(pauli_terms):
    G = 0
    q = len(pauli_terms[0][1])
    for _, pauli in pauli_terms:
        locality = q-pauli.count("I")
        G += 2*(locality) - 3
    return G

#_XY_multiP1_ij_string(4, 2, 3, [0, 1], 1)
m = 3
n = 2
pauli_terms = n_par_linear_scaling(m, n)

'''
twoq_terms = 0
for coeff, pauli in pauli_terms:
    if 18-pauli.count("I") == 2:
        twoq_terms += 1
print(twoq_terms)
'''

print("#pauli terms =", len(pauli_terms))
#print(pauli_terms)
simple_plot(pauli_terms, hamming_k=n, partition_weights=[1]*n, curve_step=50)

#pauli terms = 12


HTML(value="<div style='font-family:system-ui,Segoe UI,Arial; font-size:14px; line-height:1.7'><b>Legend (Paul…

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'randomize': False, 'animate': False, 'numIter': 4000, 'cool…

# Drawing

In [1]:
# !pip install ipycytoscape ipywidgets
from ipycytoscape import CytoscapeWidget, Node, Edge
import ipywidgets as w
import json, time

# -----------------------------
# Widget setup (no cy.on used)
# -----------------------------
cy = CytoscapeWidget()
cy.set_layout(name='preset')  # keep manual positions

cy.set_style([
    {'selector': 'node', 'style': {
        'label': 'data(id)', 'width': 34, 'height': 34,
        'text-valign': 'center', 'text-halign': 'center'
    }},
    {'selector': 'edge', 'style': {
        'curve-style': 'bezier',
        'target-arrow-shape': 'triangle',
        'label': 'data(label)',
        'text-rotation': 'autorotate',
        'font-size': 12,
        'text-background-color': '#ffffff',
        'text-background-opacity': 0.7,
        'text-background-padding': 2
    }},
    {'selector': ':selected', 'style': {'border-width': 4, 'border-color': '#f39c12'}}
])

# -----------------------------
# UI controls
# -----------------------------
mode       = w.ToggleButtons(options=['Select/Move', 'Edge mode'], value='Select/Move', description='Mode:')
node_id    = w.Text(placeholder='(auto)', description='Node ID:')
add_btn    = w.Button(description='Add node', button_style='success')
del_btn    = w.Button(description='Delete selected', button_style='danger')
export_btn = w.Button(description='Export', icon='download')
status     = w.HTML('<i>In “Edge mode”, click one node, then another, to add an edge.</i>')

# Edge label popup
label_in   = w.Text(placeholder='edge label…', description='Label:')
apply_btn  = w.Button(description='Apply', button_style='info')
label_box  = w.HBox([label_in, apply_btn])
label_box.layout.display = 'none'

# Optional dropdown fallback panel
src_dd     = w.Dropdown(options=[], description='u:')
tgt_dd     = w.Dropdown(options=[], description='v:')
mk_edge_btn= w.Button(description='Make edge')

# -----------------------------
# State
# -----------------------------
next_id        = {'n': 1}
pending_src    = {'id': None}      # first node clicked in Edge mode
last_edge_ref  = {'edge': None}    # edge awaiting label

# -----------------------------
# Helpers
# -----------------------------
def get_node_ids():
    return [n.data['id'] for n in cy.graph.nodes]

def refresh_status(msg):
    status.value = msg

def refresh_dropdowns():
    ids = get_node_ids()
    src_dd.options = ids
    tgt_dd.options = ids

# -----------------------------
# Graph ops
# -----------------------------
def add_node(new_id=None, pos=None):
    if not new_id or new_id.strip() == '':
        while True:
            cand = f"n{next_id['n']}"; next_id['n'] += 1
            if cand not in get_node_ids():
                new_id = cand; break
    if pos is None:
        pos = {'x': 0, 'y': 0}
    cy.graph.add_node(Node(data={'id': new_id}, position=pos))
    refresh_status(f'Added node <b>{new_id}</b>.')
    refresh_dropdowns()

def add_edge(u, v):
    if u == v:
        refresh_status('Ignored self-edge.'); return None
    existing = {(e.data['source'], e.data['target']) for e in cy.graph.edges}
    # Treat as undirected duplicate; remove next line to allow both directions
    existing |= {(t, s) for (s, t) in list(existing)}
    if (u, v) in existing:
        refresh_status(f'Edge {u}–{v} already exists.'); return None
    e = Edge(data={'source': u, 'target': v, 'label': ''})
    cy.graph.add_edge(e)
    refresh_status(f'Added edge <b>{u}</b> → <b>{v}</b>. Enter a label…')
    return e

def delete_selection(_=None):
    for e in list(cy.selected_edges):
        cy.graph.remove_edge(e)
    for n in list(cy.selected_nodes):
        cy.graph.remove_node(n)
    refresh_status('Deleted selected elements.')
    refresh_dropdowns()

def export_graph(_=None):
    nodes = [{'id': n.data['id'], 'position': n.position} for n in cy.graph.nodes]
    edges = [{'source': e.data['source'], 'target': e.data['target'], 'label': e.data.get('label','')} for e in cy.graph.edges]
    fn = f'graph_{int(time.time())}.json'
    with open(fn, 'w', encoding='utf-8') as f:
        json.dump({'nodes': nodes, 'edges': edges}, f, indent=2)
    refresh_status(f'Exported to <b>{fn}</b>.')

# -----------------------------
# Labeling
# -----------------------------
def show_label_prompt(edge):
    last_edge_ref['edge'] = edge
    label_in.value = ''
    label_box.layout.display = ''
    label_in.focus()

def apply_label(_=None):
    e = last_edge_ref.get('edge')
    if e is None:
        label_box.layout.display = 'none'
        return
    e.data['label'] = label_in.value
    label_box.layout.display = 'none'
    refresh_status(f'Set label to <b>{e.data["label"]}</b>.')

apply_btn.on_click(apply_label)
label_in.on_submit(lambda _: apply_label())

# -----------------------------
# Trait-based interaction (no cy.on)
#   - In Edge mode: when selection changes, use the latest two distinct node IDs
# -----------------------------
def on_selection_changed(change):
    if change['name'] != 'selected_nodes':
        return
    if mode.value != 'Edge mode':
        return
    sel = [n.data['id'] for n in cy.selected_nodes]
    if not sel:
        return
    # first click picks source
    if pending_src['id'] is None:
        pending_src['id'] = sel[-1]
        refresh_status(f'Edge start at <b>{pending_src["id"]}</b>. Now click a target node.')
        return
    # second click picks target (different from source)
    src = pending_src['id']
    tgt = sel[-1]
    if tgt == src:
        refresh_status('Pick a different target node.')
        return
    pending_src['id'] = None
    e = add_edge(src, tgt)
    if e:
        show_label_prompt(e)
    # clear selection for a clean next pair
    try:
        cy.unselect_all()
    except Exception:
        pass

cy.observe(on_selection_changed, names='selected_nodes')

# -----------------------------
# Buttons & fallback panel
# -----------------------------
add_btn.on_click(lambda _: add_node(node_id.value or None))
del_btn.on_click(delete_selection)
export_btn.on_click(export_graph)

def make_edge_from_dropdown(_=None):
    u, v = src_dd.value, tgt_dd.value
    if u and v:
        e = add_edge(u, v)
        if e:
            show_label_prompt(e)

mk_edge_btn.on_click(make_edge_from_dropdown)
refresh_dropdowns()

# -----------------------------
# Show UI
# -----------------------------
w.VBox([
    mode,
    w.HBox([node_id, add_btn, del_btn, export_btn]),
    w.HBox([src_dd, tgt_dd, mk_edge_btn]),  # remove if you don't want the dropdown fallback
    cy,
    label_box,
    status
])

/home/jchen9/pauli_first_quantization/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))
/tmp/ipykernel_3064610/210533231.py:135: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  label_in.on_submit(lambda _: apply_label())


In [34]:
# Left-click to create a node (near the clicked node), using only traitlets.
# No cy.on(...), no ipyevents—works in environments that block mouse events.

from ipycytoscape import CytoscapeWidget, Node, Edge
import ipywidgets as w
import json, time
import math

# --- widget ---
cy = CytoscapeWidget()
cy.set_layout(name='preset')  # positions stick when you drag

cy.set_style([
    {'selector': 'node', 'style': {
        'label': 'data(id)', 'width': 34, 'height': 34,
        'text-valign': 'center', 'text-halign': 'center'
    }},
    {'selector': 'edge', 'style': {
        'curve-style': 'bezier', 'target-arrow-shape': 'triangle',
        'label': 'data(label)', 'text-rotation': 'autorotate',
        'font-size': 12, 'text-background-color': '#fff',
        'text-background-opacity': 0.7, 'text-background-padding': 2
    }},
    {'selector': ':selected', 'style': {'border-width': 4, 'border-color': '#f39c12'}}
])

# --- UI ---
mode = w.ToggleButtons(
    options=['Select/Move', 'Add-node mode', 'Edge mode'],
    value='Select/Move',
    description='Mode:'
)
del_btn    = w.Button(description='Delete selected', button_style='danger')
export_btn = w.Button(description='Export', icon='download')
status     = w.HTML('<i>In “Add-node mode”, LEFT-click a node to spawn a new node next to it. In “Edge mode”, click two nodes to connect them.</i>')

label_in   = w.Text(placeholder='edge label…', description='Label:')
apply_btn  = w.Button(description='Apply', button_style='info')
label_box  = w.HBox([label_in, apply_btn]); label_box.layout.display = 'none'

# fallback dropdowns for edges
src_dd, tgt_dd = w.Dropdown(options=[], description='u:'), w.Dropdown(options=[], description='v:')
mk_edge_btn    = w.Button(description='Make edge')

# --- state ---
next_id = {'n': 1}
pending_src = {'id': None}
last_edge = {'e': None}
spawn_theta = {'t': 0}  # spiral angle to stagger new nodes around the clicked node

# --- helpers ---
def say(msg): status.value = msg
def node_ids(): return [n.data['id'] for n in cy.graph.nodes]

def refresh_node_dd():
    ids = node_ids()
    src_dd.options = ids
    tgt_dd.options = ids

def new_id():
    while True:
        cand = f"n{next_id['n']}"; next_id['n'] += 1
        if cand not in node_ids():
            return cand

def add_node_at(x, y, nid=None):
    nid = nid or new_id()
    cy.graph.add_node(Node(data={'id': nid}, position={'x': float(x), 'y': float(y)}))
    refresh_node_dd()
    say(f'Added node <b>{nid}</b>.')
    return nid

def add_node_near(base_node, radius=50):
    # Place new node on a small spiral around base_node to avoid overlaps
    p = base_node.position or {'x': 0, 'y': 0}
    theta = spawn_theta['t']
    dx = radius * math.cos(theta)
    dy = radius * math.sin(theta)
    spawn_theta['t'] = (theta + math.pi/4) % (2*math.pi)
    return add_node_at(p.get('x', 0) + dx, p.get('y', 0) + dy)

def add_edge(u, v):
    if u == v:
        say('Ignored self-edge.'); return None
    existing = {(e.data['source'], e.data['target']) for e in cy.graph.edges}
    existing |= {(b, a) for (a, b) in list(existing)}  # treat undirected duplicates as same
    if (u, v) in existing:
        say(f'Edge {u}–{v} already exists.'); return None
    e = Edge(data={'source': u, 'target': v, 'label': ''})
    cy.graph.add_edge(e)
    say(f'Added edge <b>{u}</b> → <b>{v}</b>. Enter a label…')
    return e

def show_label(e):
    last_edge['e'] = e
    label_in.value = ''
    label_box.layout.display = ''
    label_in.focus()

def apply_label(_=None):
    e = last_edge.get('e')
    if e is None:
        label_box.layout.display = 'none'; return
    e.data['label'] = label_in.value
    label_box.layout.display = 'none'
    say(f'Set label to <b>{e.data["label"]}</b>.')
apply_btn.on_click(apply_label)
label_in.on_submit(lambda _: apply_label())

# --- trait-based interaction ---
def on_selection_changed(change):
    if change['name'] != 'selected_nodes':
        return
    sel_nodes = list(cy.selected_nodes)
    if not sel_nodes:
        return
    clicked = sel_nodes[-1]          # node you just left-clicked

    if mode.value == 'Add-node mode':
        # Left-click on a node -> spawn a new node adjacent to it
        new_nid = add_node_near(clicked)
        try: cy.unselect_all()
        except Exception: pass
        return

    if mode.value == 'Edge mode':
        tgt_id = clicked.data['id']
        if pending_src['id'] is None:
            pending_src['id'] = tgt_id
            say(f'Edge start at <b>{tgt_id}</b>. Now click a target node.')
        else:
            src = pending_src['id']; pending_src['id'] = None
            if src == tgt_id:
                say('Pick a different target node.')
            else:
                e = add_edge(src, tgt_id)
                if e: show_label(e)
            try: cy.unselect_all()
            except Exception: pass

cy.observe(on_selection_changed, names='selected_nodes')

# --- buttons ---
def on_delete(_):
    for e in list(cy.selected_edges): cy.graph.remove_edge(e)
    for n in list(cy.selected_nodes): cy.graph.remove_node(n)
    say('Deleted selected.')
    refresh_node_dd()

def on_export(_):
    nodes = [{'id': n.data['id'], 'position': n.position} for n in cy.graph.nodes]
    edges = [{'source': e.data['source'], 'target': e.data['target'], 'label': e.data.get('label','')} for e in cy.graph.edges]
    fn = f'graph_{int(time.time())}.json'
    with open(fn, 'w', encoding='utf-8') as f: json.dump({'nodes': nodes, 'edges': edges}, f, indent=2)
    say(f'Exported to <b>{fn}</b>.')

def on_dropdown_edge(_):
    u, v = src_dd.value, tgt_dd.value
    if u and v:
        e = add_edge(u, v)
        if e: show_label(e)

del_btn.on_click(on_delete)
export_btn.on_click(on_export)
mk_edge_btn.on_click(on_dropdown_edge)

# seed one node so you have something to click
add_node_at(0, 0, nid='n0')

w.VBox([
    mode,
    w.HBox([del_btn, export_btn]),
    w.HBox([src_dd, tgt_dd, mk_edge_btn]),
    cy,
    label_box,
    status
])

/tmp/ipykernel_3453982/2148168643.py:108: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  label_in.on_submit(lambda _: apply_label())


In [183]:
# !pip install ipycytoscape ipywidgets
from ipycytoscape import CytoscapeWidget, Graph, Node, Edge
from ipywidgets import VBox, HBox, Button, Output

# -----------------------
# Persistent app state
# -----------------------
nodes = []          # [{'id': 'n0', 'label': '0', 'x': float, 'y': float}, ...]
edges = []          # [{'source': 'n0', 'target': 'n1', 'label': ''}, ...]
next_uid = 0        # unique id counter for stable node ids
pending_src = None  # first-click buffer for edge creation
ALLOW_SELF = True
SPACING = 140       # horizontal spacing for new nodes
BASE_X = 140
BASE_Y  = 140

out = Output()

# -----------------------
# Helpers
# -----------------------
def node_id_from(obj):
    if hasattr(obj, 'data') and isinstance(obj.data, dict):
        return obj.data.get('id')
    if isinstance(obj, dict):
        d = obj.get('data')
        if isinstance(d, dict):
            return d.get('id')
        return obj.get('id')
    return None

def undirected_edge_exists(u, v):
    return any((e['source']==u and e['target']==v) or (e['source']==v and e['target']==u) for e in edges)

def sync_positions_from_widget(cy_widget):
    """Copy current positions from live widget into nodes[] (called on tapend, and before rebuilds)."""
    if not cy_widget or not getattr(cy_widget, "graph", None):
        return
    pos_map = {}
    for n in cy_widget.graph.nodes:
        nid = n.data.get('id')
        pos = n.position or {}
        if nid is not None:
            pos_map[nid] = (float(pos.get('x', 0.0)), float(pos.get('y', 0.0)))
    if not pos_map:
        return
    for nd in nodes:
        if nd['id'] in pos_map:
            nd['x'], nd['y'] = pos_map[nd['id']]

def relabel_consecutive_left_to_right():
    """Relabel nodes' 'label' to 0..N-1 based on current x (then y) order."""
    ordering = sorted(nodes, key=lambda n: (n['x'], n['y']))
    label_map = {n['id']: str(i) for i, n in enumerate(ordering)}
    for nd in nodes:
        nd['label'] = label_map[nd['id']]

def add_node_data():
    """Append a node to the right; label is next integer."""
    global next_uid
    nid = f"n{next_uid}"; next_uid += 1
    idx = len(nodes)
    nodes.append({'id': nid, 'label': str(idx), 'x': float(BASE_X + idx*SPACING), 'y': float(BASE_Y)})
    with out: print(f"added node {nid} (label {idx}) at ({int(BASE_X + idx*SPACING)}, {int(BASE_Y)})")

def add_edge_data(u, v):
    if (not ALLOW_SELF) and u == v:
        with out: print("ignored self-edge"); return
    if undirected_edge_exists(u, v):
        with out: print(f"edge {u}—{v} already exists"); return
    ids = {n['id'] for n in nodes}
    if u not in ids or v not in ids:
        return
    edges.append({'source': u, 'target': v, 'label': ''})
    with out: print(f"added edge: {u} — {v}")

def delete_node_data(nid):
    """Remove node and incident edges; then relabel remaining nodes consecutively."""
    global pending_src
    nodes[:] = [n for n in nodes if n['id'] != nid]
    edges[:] = [e for e in edges if e['source'] != nid and e['target'] != nid]
    if pending_src == nid:
        pending_src = None
    relabel_consecutive_left_to_right()
    with out: print(f"removed node {nid}; relabeled remaining nodes")

# -----------------------
# Widget (re)builder
# -----------------------
def build_cy():
    cy = CytoscapeWidget()
    cy.set_layout(name='preset', fit=False)  # respect positions; no auto-fit
    cy.set_style([
        {'selector': 'node', 'style': {
            'label': 'data(label)', 'width': 34, 'height': 34,
            'text-valign': 'center', 'text-halign': 'center'
        }},
        {'selector': 'edge', 'style': {
            'curve-style': 'bezier', 'label': 'data(label)',
            'control-point-step-size': 40, 'loop-direction': '-45deg', 'loop-sweep': '40deg',
        }},
        {'selector': ':selected', 'style': {'border-width': 4, 'border-color': '#f39c12'}}
    ])

    g = Graph(multigraph=False)
    for n in nodes:
        g.add_node(Node(data={'id': n['id'], 'label': n['label']},
                        position={'x': n['x'], 'y': n['y']}))
    for e in edges:
        g.add_edge(Edge(data={'source': e['source'], 'target': e['target'], 'label': e['label']}))
    cy.graph = g

    # --- Handlers using only the allowed events ---

    # Capture drag moves when the user finishes a tap (mouseup/touchend)
    
    # assumes: cy is a CytoscapeWidget with layout='preset' and fit=False
    out = Output()
    display(out)

    start_pos = {}   # nid -> (x0, y0)
    moved = set()    # nids seen during tapdrag

    def node_id_from(obj):
        if hasattr(obj, 'data') and isinstance(obj.data, dict):
            return obj.data.get('id')
        if isinstance(obj, dict):
            d = obj.get('data')
            return d.get('id') if isinstance(d, dict) else obj.get('id')
        return None

    def get_node_xy(cy_widget, nid):
        for n in cy_widget.graph.nodes:
            if n.data.get('id') == nid:
                pos = n.position or {}
                return float(pos.get('x', 0.0)), float(pos.get('y', 0.0))
        return None

    # record start
    def on_tapstart(node):
        nid = node_id_from(node)
        if not nid: return
        xy = get_node_xy(cy, nid)
        if xy is not None:
            start_pos[nid] = xy

    # mark as moved while dragging (optional but nice)
    def on_tapdrag(node):
        nid = node_id_from(node)
        if nid: moved.add(nid)

    # on release, report nodes that actually changed position
    def on_tapend(node):
        nid = node_id_from(node)
        if not nid: return
        xy0 = start_pos.pop(nid, None)
        xy1 = get_node_xy(cy, nid)
        if xy1 is None: return
        if xy0 is None or xy0 != xy1 or nid in moved:
            with out:
                print(f"moved node {nid}: x={xy1[0]:.1f}, y={xy1[1]:.1f}")
        moved.discard(nid)

    # wire events (only using the input/higher-level events you listed)
    cy.on('node', 'tapstart', on_tapstart)  # start of drag/press
    cy.on('node', 'tapdrag',  on_tapdrag)   # during drag (optional)
    cy.on('node', 'tapend',   on_tapend)    # end of drag/release

    # Two-click edge creation (tap)
    def on_node_tap(node):
        global pending_src
        nid = node_id_from(node)
        if nid is None:
            return
        if pending_src is None:
            pending_src = nid
            with out: print(f"start edge at: {nid}")
        else:
            src = pending_src
            pending_src = None
            # ensure we have the latest positions before mutating & rebuilding
            sync_positions_from_widget(cy)
            add_edge_data(src, nid)
            rebuild_cy()

    # Right-click (cxttap) delete
    def on_node_cxttap(node):
        nid = node_id_from(node)
        if not nid:
            return
        # sync positions before mutating
        sync_positions_from_widget(cy)
        delete_node_data(nid)
        rebuild_cy()

    cy.on('node', 'tap',     on_node_tap)     # normalized click/tap
    cy.on('node', 'cxttap',  on_node_cxttap)  # right-click / two-finger tap
    return cy

def rebuild_cy():
    """Sync from current widget, rebuild a fresh widget, and swap it into the VBox."""
    try:
        current = root_box.children[1]
        sync_positions_from_widget(current)
    except Exception:
        pass
    new_cy = build_cy()
    children = list(root_box.children)
    children[1] = new_cy  # [controls, cy, out]
    root_box.children = tuple(children)

# -----------------------
# Controls & layout
# -----------------------
add_btn = Button(description="Add node", button_style="success")
def on_add(_):
    # sync positions in case user just dragged
    try:
        current = root_box.children[1]
        sync_positions_from_widget(current)
    except Exception:
        pass
    add_node_data()
    rebuild_cy()
add_btn.on_click(on_add)

initial_cy = build_cy()
root_box = VBox([HBox([add_btn]), initial_cy, out])
root_box

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

In [1]:
# !pip install ipycytoscape ipywidgets
from ipycytoscape import CytoscapeWidget, Node, Edge
from ipywidgets import VBox, HBox, Button, Output

# -------------------------
# Widget & styling
# -------------------------
cy = CytoscapeWidget()
cy.set_layout(name='preset', fit=False)  # keep manual positions; no auto-fit
cy.set_style([
    {'selector': 'node', 'style': {
        'label': 'data(label)', 'width': 34, 'height': 34,
        'text-valign': 'center', 'text-halign': 'center'
    }},
    {'selector': 'edge', 'style': {
        'curve-style': 'bezier', 'label': 'data(label)'
    }},
    {'selector': ':selected', 'style': {'border-width': 4, 'border-color': '#f39c12'}},
    # Soft-hide support:
    {'selector': 'node[hidden = "true"]', 'style': {'display': 'none'}},
    {'selector': 'edge[hidden = "true"]', 'style': {'display': 'none'}},
])

out = Output()

# -------------------------
# State & helpers
# -------------------------
SPACING   = 140
BASE_X    = 140
BASE_Y    = 140
ALLOW_SELF = True

pending_src = {'id': None}  # two-click buffer
next_uid    = {'i': 0}      # stable unique IDs: v0, v1, ...

def node_id_from(obj):
    if hasattr(obj, 'data') and isinstance(obj.data, dict):
        return obj.data.get('id')
    if isinstance(obj, dict):
        d = obj.get('data')
        return d.get('id') if isinstance(d, dict) else obj.get('id')
    return None

def visible_nodes():
    return [n for n in cy.graph.nodes if n.data.get('hidden') != "true"]

def visible_edges():
    return [e for e in cy.graph.edges if e.data.get('hidden') != "true"]

def undirected_edge_exists(u, v):
    pairs = {(e.data['source'], e.data['target']) for e in visible_edges()}
    return (u, v) in pairs or (v, u) in pairs

def relabel_visible_nodes_left_to_right():
    """Relabel visible nodes' labels to 0..N-1 in left→right (then y) order."""
    def xy(n):
        pos = n.position or {}
        return (float(pos.get('x', 0.0)), float(pos.get('y', 0.0)))
    vis_sorted = sorted(visible_nodes(), key=xy)
    for i, n in enumerate(vis_sorted):
        n.data['label'] = str(i)

# -------------------------
# Add node (button): left→right placement, label is next consecutive
# -------------------------
def add_node_horizontal(_=None):
    # label is count of visible nodes (consecutive)
    label = str(len(visible_nodes()))
    vid   = f"v{next_uid['i']}"; next_uid['i'] += 1
    x     = BASE_X + len(visible_nodes()) * SPACING
    y     = BASE_Y
    cy.graph.add_node(Node(
        data={'id': vid, 'label': label},
        position={'x': float(x), 'y': float(y)}
    ))
    with out: print(f"added node {vid} (label {label}) at ({int(x)},{int(y)})")

add_btn = Button(description="Add node", button_style="success")
add_btn.on_click(add_node_horizontal)

# -------------------------
# Click two nodes -> add undirected edge (no duplicates)
# -------------------------
def add_edge(u, v):
    if (not ALLOW_SELF) and (u == v):
        with out: print("ignored self-edge"); return
    if undirected_edge_exists(u, v):
        with out: print(f"edge {u}—{v} already exists"); return
    keep = {n.data['id'] for n in visible_nodes()}
    if u not in keep or v not in keep:
        return
    cy.graph.add_edge(Edge(data={'source': u, 'target': v, 'label': ''}))
    with out: print(f"added edge: {u} — {v}")

def on_node_click(node):
    nid = node_id_from(node)
    # ignore hidden nodes
    if nid is None or any(n.data['id']==nid and n.data.get('hidden')=="true" for n in cy.graph.nodes):
        return
    if pending_src['id'] is None:
        pending_src['id'] = nid
        with out: print(f"start edge at: {nid}")
    else:
        src = pending_src['id']; pending_src['id'] = None
        add_edge(src, nid)

cy.on('node', 'click', on_node_click)

# -------------------------
# Right-click -> soft delete node + incident edges; relabel visible nodes 0..N-1
# -------------------------
def hide_node_and_incident_edges(nid):
    # hide node
    for n in cy.graph.nodes:
        if n.data.get('id') == nid:
            n.data['hidden'] = "true"
            break
    # hide its edges
    for e in cy.graph.edges:
        if e.data.get('source') == nid or e.data.get('target') == nid:
            e.data['hidden'] = "true"

def on_node_cxttap(node):
    nid = node_id_from(node)
    if not nid:
        return
    if pending_src['id'] == nid:
        pending_src['id'] = None
    hide_node_and_incident_edges(nid)
    relabel_visible_nodes_left_to_right()
    with out: print(f"removed node {nid} (soft); relabeled remaining nodes 0..{len(visible_nodes())-1}")

cy.on('node', 'cxttap', on_node_cxttap)

# -------------------------
# UI
# -------------------------
VBox([HBox([add_btn]), cy, out])

/home/jchen9/pauli_first_quantization/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))
